# Proyecto 1 — EEG con parpadeos reales (Eye-BCI): sacar el EOG con NLMS

**Cómo leer este notebook:** igual que `proyecto1_g4.ipynb`. Cada sección empieza con un recuadro **"Dónde estamos"**. Antes de cada celda de código hay un bloque que dice qué hace y por qué. Debajo de cada gráfico o tabla hay un **"Qué mirar"** con lo que se ve y los números medidos.

---

## Por qué otro dataset
En ds008701 el EEG venía **preprocesado**: le habían sacado los parpadeos. El ruido de ojos que quedaba era chico (movimientos horizontales), así que no podíamos probar el filtro contra el ruido de ojos más importante.

**Eye-BCI** (Guttmann-Flury, Sheng & Zhu 2025, *Scientific Data*, DOI 10.1038/s41597-025-04861-9; copia en Zenodo, DOI 10.5281/zenodo.18971758) es **EEG crudo**, con:
- parpadeos reales en el EEG;
- un canal de EOG medido (`HEO`);
- una columna `Blinks` que marca el pico de cada parpadeo (los autores los detectaron con video y con el EEG). **Esto es lo nuevo: sabemos dónde están los parpadeos sin detectarlos nosotros.**

Usamos un solo archivo: sujeto 1, sesión 1 (`ME011`). La tarea es de **ejecución motora**: 40 repeticiones de "mirar una cruz 2 s → agarrar con la mano izquierda o derecha 4 s → descansar 1–1.5 s".

## El modelo (el mismo de antes, pero solo con EOG)
Lo que mide un electrodo:

`y(n) = s(n) + λ·η_EOG(n)`

- `s(n)`: el EEG, lo que queremos.
- `η_EOG(n)`: el ruido de los ojos. `λ` dice cuánto llega a ese electrodo.

La referencia es el canal `HEO`, y el filtro FIR la convierte en "el parpadeo tal como llega al electrodo":

`x(n) = [HEO(n), HEO(n−1), HEO(n−2)]ᵀ`,  `ŷ(n) = wᵀ(n) x(n)`,  `e(n) = y(n) − ŷ(n) ≈ s(n)`

```
 s(n) EEG ───────────────────┐
 η_EOG ──λ───────────────────┼──►  y(n)  (electrodo)
                             │       │
 x(n) = HEO ──► [ w ] ── ŷ(n) ──────►(−)──► e(n) = y − ŷ ≈ s(n)
                  ▲                          │
                  └─────── NLMS ◄────────────┘
```

**Dos decisiones de arranque:**
- **Solo una referencia, el EOG.** El archivo no tiene canal de EMG. El EMG se discute después de ver los resultados.
- **Solo NLMS.** VS-LMS y la compuerta de §8 del notebook anterior salieron de conocer *esa* señal. Acá primero miramos qué hace NLMS tal cual.

## Mapa del notebook

| Sección | Pregunta que responde |
|---|---|
| §1 Carga | ¿Cómo leo el `.bdf` y el `_sync.csv`, y cómo los pongo en el mismo reloj? |
| §2 Inspección | ¿Qué ruido hay realmente? ¿Cuánto parpadeo llega a cada canal? ¿`HEO` sirve de referencia? |
| §3 El filtro | NLMS y Wiener, chequeos de cordura y una primera corrida sobre los datos reales. |
| §4 Diagnóstico | ¿Por qué NLMS, tal como estaba, no anda con estos datos? |
| §5 Semi-sintético | Con una mezcla armada donde conocemos la respuesta: ¿qué algoritmo, μ̃, δ y M? |
| §6 Datos reales | Con lo elegido, ¿el filtro saca el parpadeo de la grabación cruda sin dañar el EEG? |
| §7 Conclusiones | ¿Sirvió? |

**Resumen** (el detalle y los números están en §7): **sí sirvió, pero no con NLMS tal como lo veníamos usando.** Con `δ` chico, NLMS da sus pasos más grandes entre parpadeos, cuando la referencia no trae ruido, y los coeficientes nunca se quedan quietos. **Con un `δ` grande, NLMS se porta como LMS entre parpadeos y solo normaliza en los parpadeos más grandes.** Así, sobre los datos crudos, saca el ~90 % del parpadeo de `Fp1`/`Fp2`, queda casi igual que el mejor filtro fijo (Wiener) y no toca el canal de control.

## §1. Carga

> **Dónde estamos:** antes de filtrar hay que leer tres archivos y ponerlos en el mismo reloj.
> - `ME011.bdf`: el EEG. BDF es el formato de BioSemi: como EDF, pero con muestras de 24 bits. MNE lo lee con `read_raw_bdf`.
> - `ME011_sync.csv`: una fila por milisegundo. Trae la columna `Blinks` (1 = pico de parpadeo) y `Cues` (en qué parte de la tarea está).
> - `ME011_annotations.json`: el cronograma de cada repetición. Lo usamos solo para chequear.
>
> Pasos: parámetros → carga y alineación → chequeo de lo que se cargó.

### 1.1 Parámetros
- **`FS = 250`:** bajamos de 1000 a 250 Hz, como antes. Todo lo que nos importa (EEG y EOG) está por debajo de ~40 Hz, y el filtro corre 4 veces más rápido.
- **`T_INI, T_FIN = 2, 338`:** la tarea termina a los ~338 s, pero la grabación sigue hasta los 452 s. En ese final la señal se rompe: el desvío de los canales pasa de ~30 µV a ~700 µV (probablemente la persona se mueve). Los primeros 2 s se sacan por el transitorio del pasa-altos.
- **`F_HP = 0.5`:** el `.bdf` viene sin filtrar (la cabecera dice pasa-altos 0 Hz), con continua y derivas de miles de µV. **El mismo pasa-altos se aplica a todos los canales, a `y` y a `x` por igual.** Si filtrás distinto la deseada y la referencia, cambiás el camino que el filtro tiene que aprender.
- **`CANALES_MALOS`:** los 11 canales que el artículo declara defectuosos o puenteados. No los usamos. Ojo: incluye `Oz`, el canal que usábamos antes como "EEG limpio".

In [ ]:
import json
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, lfilter, welch, correlate

mne.set_log_level("ERROR")

# Parámetros
RAIZ = "./datasets-presentacion-PASA/Sess01"
ARCHIVO = "ME011"           # Sujeto 01, sesión 1
FS_ORIG = 1000              # fs del .bdf [Hz]
FS = 250                    # Submuestreo 1000 -> 250 Hz
T_INI, T_FIN = 2, 338       # Ventana útil [s]: la tarea termina a los ~338 s y después la señal se rompe
F_HP = 0.5                  # Pasa-altos [Hz]: saca la continua y la deriva, igual en todos los canales
CANAL_EOG = "HEO"
CANALES_MALOS = ["PO3", "F1", "POz", "Oz", "F3", "O2", "P8", "PO7", "FC3", "P7", "P4"]  # Defectuosos según el artículo

### 1.2 Carga y alineación
`cargar()` hace lo mismo que en el notebook anterior: pasa-altos → submuestreo → recorte → µV → sin continua. Además, antes de descartar el canal `Trig`, lee de ahí cuándo empieza cada repetición.

Después lee `_sync.csv`:
- **Cómo se alinea:** la columna `Time` está en el mismo reloj que el `.bdf`. Por ejemplo, el primer `Fixation` está en `Time = 24.398` y el primer disparo del `.bdf` en la muestra 24 398. Entonces un parpadeo en `Time = t_p` es la muestra `(t_p − T_INI)·FS` de nuestras señales.
- **Sobran filas:** el `_sync.csv` tiene 560 filas más que el `.bdf` (452.56 s contra 452 s). Están al final, después de nuestro recorte, así que no molestan.
- **`Cues`** dice la fase de la tarea, pero viene salpicada: una etiqueta cada pocos ms y vacíos entre medio. Se completa hacia adelante y se traduce a fijación, ejecución (`Left`/`Right`), descanso (`Random`) y "otra" (bienvenida y despedida).

Por último, compara los `onset` de `_annotations.json` con los disparos del `.bdf`.

In [ ]:
def cargar():
    """
    Carga el .bdf. Devuelve:
        datos: dict {canal: señal en µV}, con pasa-altos F_HP, submuestreada a FS y recortada a [T_INI, T_FIN]
        t_disparos: instantes [s] en que el canal Trig marca el comienzo de una repetición
        duracion: duración total del .bdf [s]
    """
    raw = mne.io.read_raw_bdf(f"{RAIZ}/{ARCHIVO}.bdf", preload=True, stim_channel=None)
    trig = raw.get_data(picks=["Trig"])[0]
    t_disparos = np.flatnonzero(np.diff(trig) > 0) / FS_ORIG   # Flancos de subida del disparo
    raw.drop_channels(["Trig"])               # El disparo no es una señal
    raw.filter(F_HP, None)                    # Mismo pasa-altos para y y para x
    raw.resample(FS)                          # MNE aplica el antialias
    raw.crop(T_INI, T_FIN)
    X = raw.get_data() * 1e6                  # V -> µV
    X = X - X.mean(axis=1, keepdims=True)     # Sin continua
    return dict(zip(raw.ch_names, X)), t_disparos, len(trig) / FS_ORIG


DATOS, t_disparos, DURACION = cargar()
N_MUESTRAS = len(DATOS[CANAL_EOG])
t = T_INI + np.arange(N_MUESTRAS) / FS        # Tiempo de cada muestra [s], en el reloj del .bdf

# _sync.csv: una fila por ms. Time es el tiempo en el reloj del .bdf
sync = pd.read_csv(f"{RAIZ}/{ARCHIVO}_sync.csv", usecols=["Time", "Cues", "Blinks"], dtype={"Cues": str})
t_parpadeos = sync["Time"].values[sync["Blinks"].values == 1]                     # Picos de parpadeo [s]
t_parpadeos = t_parpadeos[(t_parpadeos > T_INI) & (t_parpadeos < T_FIN)]           # Solo los del recorte
PARPADEOS = np.round((t_parpadeos - T_INI) * FS).astype(int)                       # Índices a FS

# Fase de la tarea en cada muestra: Cues viene salpicado cada pocos ms, se completa hacia adelante
NOMBRE_FASE = {"Welcome": "otra", "Goodbye": "otra", "Fixation": "fijación",
               "Left": "ejecución", "Right": "ejecución", "Random": "descanso"}
cues = sync["Cues"].ffill().fillna("Welcome").values
FASE = np.array([NOMBRE_FASE[c] for c in cues[np.searchsorted(sync["Time"].values, t)]])

# _annotations.json: cronograma de cada repetición, para comparar con los disparos del .bdf
anot = pd.DataFrame(json.load(open(f"{RAIZ}/{ARCHIVO}_annotations.json")))
onset_fijacion = anot.loc[anot["phase"] == "fixation", "onset"].values

print(f"{len(DATOS)} canales: {list(DATOS)}")
print(f"{N_MUESTRAS / FS:.0f} s a {FS} Hz (recorte {T_INI}-{T_FIN} s)")
print(f"_sync.csv: {len(sync)} filas = {sync['Time'].iloc[-1]:.2f} s; el .bdf dura {DURACION:.2f} s")
print(f"{len(t_disparos)} repeticiones; {len(PARPADEOS)} parpadeos marcados en el recorte "
      f"({len(PARPADEOS) / (N_MUESTRAS / FS) * 60:.1f} por minuto)")
print(f"annotations.json vs disparos del .bdf: onset - disparo = "
      f"{np.mean(onset_fijacion - t_disparos):.3f} ± {np.std(onset_fijacion - t_disparos):.3f} s")
print("Muestras por fase:", pd.Series(FASE).value_counts().to_dict())

### Qué mirar: lo que se cargó
- **65 canales:**
  - 62 de EEG, con los nombres del sistema 10-20 (`Fp1`, `Fz`, `Pz`…), más `CB1` y `CB2`, sobre el cerebelo;
  - `M1` y `M2`: las mastoides, detrás de las orejas;
  - `HEO`: el único canal de EOG.

  **No hay un segundo EOG ni EMG.** El artículo describe un EOG vertical y dos EMG en los párpados; en esta copia de Zenodo hay un solo EOG, con el nombre `HEO` (ver 2.2).
- **75 parpadeos en 336 s (13.4 por minuto).** En todo el archivo hay 105; los otros 30 caen en el final que recortamos.
- **40 repeticiones**, como dice el artículo.
- **`annotations.json` está corrido exactamente −2.000 s** respecto de los disparos del `.bdf`. Por eso usamos `Cues` del `_sync.csv`, que está en el reloj del `.bdf`, y no el `.json`.
- **Tiempo por fase:** la mitad es ejecución (171 s), un cuarto fijación (86 s), 56 s de descanso y 22 s de "otra".

## §2. Inspección

> **Dónde estamos:** ya tenemos las señales. Antes de filtrar hay que saber qué hay adentro: eso decide qué canal usar como `y`, si `HEO` sirve de referencia y qué esperar del filtro.
>
> Preguntas, en orden:
> 1. ¿Hay canales rotos? (tabla de desvíos)
> 2. ¿Cómo se ven los parpadeos? (señal en el tiempo)
> 3. ¿Cuánto parpadeo llega a cada canal? (promedio sincronizado)
> 4. ¿Cuánto vale `λ` en cada electrodo? (mapa)
> 5. ¿El camino ojo→electrodo tiene retardo? (correlación cruzada; decide `M`)
> 6. ¿Qué frecuencias hay? (PSD)
> 7. ¿`HEO` trae algo más que parpadeo? (fuga de EEG en la referencia)
> 8. ¿Hay ruido que no sea de los ojos? (parte común a todos los canales)
> 9. ¿Cuánta señal hay sin parpadeos? (insumo para un futuro semi-sintético)
> 10. ¿La tarea motora mete ruido? (desvío por fase)

### 2.1 ¿Hay canales rotos?
Calcula el desvío estándar de cada canal (después del pasa-altos y del recorte) y lo compara con la mediana de los canales buenos. Un canal desconectado da un desvío ~0; uno con mal contacto, muchas veces la mediana.

In [ ]:
CANALES_EEG = [n for n in DATOS if n not in ["M1", "M2", CANAL_EOG]]
CANALES_BUENOS = [n for n in CANALES_EEG if n not in CANALES_MALOS]

desvio = pd.Series({n: DATOS[n].std() for n in DATOS})
mediana = desvio[CANALES_BUENOS].median()
tabla = pd.DataFrame({"std [µV]": desvio, "std / mediana": desvio / mediana,
                      "malo según el artículo": [n in CANALES_MALOS for n in desvio.index]})

print(f"{len(CANALES_EEG)} canales de EEG, {len(CANALES_BUENOS)} buenos. Mediana del std de los buenos: {mediana:.1f} µV\n")
print("Los 8 con más std:\n", tabla.sort_values("std [µV]", ascending=False).head(8).round(2).to_string(), "\n")
print("Los 5 con menos std:\n", tabla.sort_values("std [µV]").head(5).round(2).to_string(), "\n")
print("Los declarados malos:\n", tabla.loc[CANALES_MALOS].round(2).to_string())

### Qué mirar: desvío por canal
- **Los canales de EEG andan parecido:** entre 15 y 35 µV, con mediana 22.9 µV. Los más grandes son los frontales (`Fp1`, `Fpz`, `Fp2`, `AF3`: ~1.5 veces la mediana), que es adonde más llega el parpadeo.
- **Los 11 "malos" del artículo no se distinguen por el desvío** (0.7 a 1.1 veces la mediana). El artículo dice que están *puenteados*: dos electrodos en corto por el gel. Eso no se ve en el desvío, se ve en que dos canales vecinos son casi iguales. Les creemos y no los usamos.
- **`M1` y `M2` son enormes:** 438 y 233 µV, 10 a 20 veces el EEG. Según el artículo, `M1` es la referencia, así que debería ser ~0. No sabemos qué se grabó en esos dos canales, y no los usamos.
- **`HEO` tiene 65 µV**, casi 3 veces el EEG. Es lo esperable para un canal pegado al ojo.

### 2.2 Los parpadeos a simple vista
Dibuja 60 s (de 30 a 90 s) de estos canales:
- `HEO`;
- los frontales `Fp1` y `Fp2`;
- los laterales de adelante `F7` y `F8`;
- uno de atrás, `Pz`;
- uno del costado, `T7`.

Las líneas rojas son los parpadeos que marca `Blinks`. En naranja, los 4 s de ejecución del movimiento.

In [ ]:
CANALES_VER = [CANAL_EOG, "Fp1", "Fp2", "F7", "F8", "Pz", "T7"]
T_VER = (30, 90)                                   # Tramo a mirar [s]

i0, i1 = [int((tv - T_INI) * FS) for tv in T_VER]
fig, axs = plt.subplots(len(CANALES_VER), 1, figsize=(12, 11), sharex=True)
for ax, n in zip(axs, CANALES_VER):
    ax.plot(t[i0:i1], DATOS[n][i0:i1], lw=0.6)
    ax.fill_between(t[i0:i1], 0, 1, where=FASE[i0:i1] == "ejecución", transform=ax.get_xaxis_transform(),
                    color="tab:orange", alpha=0.15, lw=0)
    for k in PARPADEOS[(PARPADEOS >= i0) & (PARPADEOS < i1)]:
        ax.axvline(t[k], color="r", lw=0.8, ls="--")
    ax.set_ylabel(f"{n}\n[µV]")
    ax.grid(True)
axs[0].set_title("Rojo: parpadeos marcados (Blinks = 1). Naranja: ejecución del movimiento")
axs[-1].set_xlabel("t [s]")
plt.tight_layout()
plt.show()

### Qué mirar: los parpadeos en el tiempo
- **Cada línea roja cae sobre un pico:** en `HEO` de ~250 a ~550 µV, en `Fp1` y `Fp2` de ~200 µV, y más chico en `F7`. **La columna `Blinks` está bien alineada con el `.bdf`.**
- **Los picos son angostos:** ~0.2 s, lo que dice el artículo.
- **`HEO` sirve como referencia del parpadeo, y probablemente no es horizontal.**
  - `HEO` es el nombre que usa Neuroscan para el EOG *horizontal*: un electrodo al costado de cada ojo, uno contra el otro. Mide los movimientos hacia los costados. Un parpadeo es vertical e igual en los dos ojos, así que en ese canal casi se cancela.
  - Pero acá `HEO` muestra los parpadeos más grandes que `Fp1` y `Fp2`, que están justo arriba de los ojos (2.3: 485 contra ~200 µV), con el mismo signo y sin retardo.
  - **Hipótesis (no comprobada):** es el EOG **vertical** que describe el artículo (electrodos arriba y abajo del ojo izquierdo, uno contra el otro), con el nombre cambiado en la copia de Zenodo. Encaja también con 2.8: no comparte el ruido común de los canales de EEG, como pasa con un canal bipolar.
  - **Consecuencia:** si es vertical, los movimientos de ojos hacia los costados casi no están en la referencia, y el filtro no los puede sacar.
  - Hay alguna deflexión que no es parpadeo (por ejemplo, −150 µV a los 34 s): puede ser un movimiento de ojos vertical.
- **Lo inesperado: ondas lentas de ~100 µV y ~1 s en *todos* los canales de EEG**, incluidos `Pz` y `T7`, sobre todo durante la ejecución (naranja). **En `HEO` no están.** No son de los ojos: si lo fueran, serían más grandes adelante que atrás y estarían en `HEO`. Se estudian en 2.8.

### 2.3 Cuánto parpadeo llega a cada canal
El mismo truco que en el notebook anterior (`proyecto1_g4.ipynb`, §2):
1. recortar ±1 s alrededor de cada parpadeo;
2. promediar todos los recortes.

Lo que está atado al parpadeo se suma en todos los recortes. El EEG no tiene nada que ver con el parpadeo, así que en el promedio tiende a 0.

**La diferencia con antes:** ahora los eventos los marca el dataset (`Blinks`), no los detectamos nosotros sobre el EOG, y son 75 en lugar de 6. Los canales van ordenados de adelante hacia atrás.

In [ ]:
VENTANA = int(1 * FS)                              # ±1 s alrededor de cada parpadeo
tau = np.arange(-VENTANA, VENTANA) / FS


def promedio_sincronizado(x, picos):
    """Promedio de x en ventanas de ±VENTANA alrededor de cada pico."""
    tramos = [x[k - VENTANA:k + VENTANA] for k in picos if VENTANA <= k < len(x) - VENTANA]
    return np.mean(tramos, axis=0)


CANALES_PROM = ["Fp1", "Fpz", "Fp2", "AF3", "F7", "Fz", "F8", "Cz", "Pz", "O1"]   # De adelante hacia atrás

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
axs[0].plot(tau, promedio_sincronizado(DATOS[CANAL_EOG], PARPADEOS), "k", lw=2, label=CANAL_EOG)
for n in CANALES_PROM:
    axs[0].plot(tau, promedio_sincronizado(DATOS[n], PARPADEOS), label=n)
    axs[1].plot(tau, promedio_sincronizado(DATOS[n], PARPADEOS), label=n)
axs[0].set_title(f"Promedio sincronizado con {len(PARPADEOS)} parpadeos")
axs[1].set_title("Lo mismo sin HEO (otra escala)")
for ax in axs:
    ax.set_xlabel("t respecto del pico marcado [s]")
    ax.set_ylabel("µV")
    ax.grid(True)
axs[1].legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

### Qué mirar: el parpadeo promedio
- **Tamaños de pico:**
  - `HEO`: 485 µV;
  - `Fp1`, `Fpz`, `Fp2`: ~200 µV;
  - `AF3`: ~150 µV; `F7`: ~85 µV; `Fz`: ~60 µV;
  - atrás (`Cz`, `Pz`, `O1`): 15–30 µV.
- **El pico cae en el mismo instante en todos los canales** (t ≈ 0): el camino ojo→electrodo no tiene retardo visible. Es lo esperable para un campo eléctrico, que a estas distancias llega instantáneamente.
- **La forma es la misma en todos:** un pulso de ~0.2 s con una cola hasta ~0.25 s. Solo cambia el tamaño. Eso es justamente el modelo `y = s + λ·η_EOG`: cada canal recibe una copia escalada.
- **El nivel antes y después del pico no es real** (−25 µV en `HEO`). Lo agrega el pasa-altos de 0.5 Hz, que a un pulso grande le pone lóbulos negativos a los costados. Está igual en `y` y en `x`, así que no cambia la relación entre ellas.

### 2.4 Mapa de λ
Divide el parpadeo promedio de cada canal por el de `HEO`, en el instante del pico. Ese cociente estima **`λ`**: qué fracción del parpadeo de la referencia llega a ese electrodo.

El mapa lo dibuja sobre la cabeza, vista desde arriba con la nariz hacia arriba. `CB1` y `CB2` quedan afuera porque no están en el montaje estándar de MNE.

In [ ]:
prom_eog = promedio_sincronizado(DATOS[CANAL_EOG], PARPADEOS)
k_pico = np.argmax(np.abs(prom_eog))               # Muestra del pico del parpadeo promedio en HEO
LAMBDA = pd.Series({n: promedio_sincronizado(DATOS[n], PARPADEOS)[k_pico] / prom_eog[k_pico]
                    for n in CANALES_BUENOS})      # λ estimado: amplitud en el canal / amplitud en HEO

canales_mapa = [n for n in CANALES_BUENOS if n not in ["CB1", "CB2"]]   # CB1/CB2 no están en el montaje estándar
info = mne.create_info(canales_mapa, FS, "eeg")
info.set_montage("standard_1005")

fig, ax = plt.subplots(figsize=(5, 5))
im, _ = mne.viz.plot_topomap(LAMBDA[canales_mapa].values, info, axes=ax, show=False, cmap="RdBu_r",
                             vlim=(-1, 1), names=canales_mapa)
fig.colorbar(im, ax=ax, shrink=0.7, label="λ = parpadeo en el canal / parpadeo en HEO")
ax.set_title(f"Cuánto parpadeo llega a cada canal (pico en t = {tau[k_pico]*1000:.0f} ms)")
plt.show()

print("λ más grandes:\n", LAMBDA.sort_values(ascending=False).head(8).round(2).to_string())
print("\nλ de algunos posteriores:\n", LAMBDA[["Cz", "Pz", "P3", "PO4", "O1"]].round(2).to_string())

### Qué mirar: el mapa de λ
- **Adelante:** `λ ≈ 0.4` en `Fp1`, `Fpz` y `Fp2`; 0.32 en `AF3`, 0.23 en `AF4`, ~0.18 en `F7`/`F5`, 0.13 en `Fz`.
- **Atrás:** `λ ≈ 0.03–0.06` (`Cz`, `Pz`, `P3`, `PO4`, `O1`). Les llega ~5 % del parpadeo de `HEO`, o sea ~20 µV.
- **El mapa es un gradiente de adelante hacia atrás, simétrico entre izquierda y derecha.** Así se ve un parpadeo: los dos ojos a la vez, en el medio.
- **Consecuencia práctica:**
  - `Fp1` y `Fp2` son los mejores candidatos para `y`: es donde más ruido hay que sacar;
  - `Pz` sirve de control, porque casi no le llega.

### 2.5 ¿El camino ojo→electrodo tiene retardo? (decide M)
Para cada parpadeo calcula la correlación cruzada normalizada entre `HEO` y un canal en ±1 s, y después promedia sobre los parpadeos.
- **Si el máximo está en lag 0,** el canal ve el parpadeo al mismo tiempo que `HEO`, y alcanza con un filtro muy corto: en el límite `M = 1`, que es una ganancia.
- **Si el máximo estuviera corrido,** haría falta un `M` más grande para cubrir ese retardo.

In [ ]:
LAG_MAX = int(0.1 * FS)                            # ±100 ms
lags = np.arange(-LAG_MAX, LAG_MAX + 1)


def correlacion_cruzada(a, b, picos):
    """Correlación cruzada normalizada entre a y b en cada ventana de parpadeo, promediada. Lag > 0: b atrasada."""
    rho = []
    for k in picos:
        if not VENTANA <= k < len(a) - VENTANA:
            continue
        ta, tb = a[k - VENTANA:k + VENTANA], b[k - VENTANA:k + VENTANA]
        ta, tb = ta - ta.mean(), tb - tb.mean()
        c = correlate(tb, ta, mode="full") / (np.linalg.norm(ta) * np.linalg.norm(tb))
        rho.append(c[len(ta) - 1 - LAG_MAX:len(ta) + LAG_MAX])
    return np.mean(rho, axis=0)


filas = {}
fig, ax = plt.subplots(figsize=(9, 4))
for n in ["Fp1", "Fp2", "F7", "F8", "Pz"]:
    rho = correlacion_cruzada(DATOS[CANAL_EOG], DATOS[n], PARPADEOS)
    ax.plot(lags / FS * 1000, rho, label=n)
    filas[n] = {"lag del máximo [ms]": lags[np.argmax(np.abs(rho))] / FS * 1000, "ρ(0)": rho[LAG_MAX],
                "ρ máx": rho[np.argmax(np.abs(rho))]}
ax.set_xlabel("lag [ms] (> 0: el canal va atrasado respecto de HEO)")
ax.set_ylabel("ρ")
ax.set_title("Correlación cruzada HEO ↔ canal en ventanas de ±1 s alrededor de cada parpadeo")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()
print(pd.DataFrame(filas).T.round(2).to_string())

### Qué mirar: correlación cruzada
- **No hay retardo:** el máximo está en 0 ms para `Fp1`, `Fp2` y `F7`, y en 4 ms (1 muestra) para `F8` y `Pz`. El camino es casi una ganancia pura.
- **ρ(0) = 0.90 en `Fp1` y `Fp2`.** Durante los parpadeos, `HEO` explica el 81 % (ρ²) de la potencia de `Fp1`. Baja a 0.74 en `F7`, 0.46 en `F8` y 0.23 en `Pz`.
- **Las curvas son anchas** (recién caen a ±100 ms) porque el parpadeo es lento comparado con la muestra (4 ms). No es retardo.
- **Para el filtro:** `M = 3` (el valor de antes, He et al. 2004) alcanza de sobra. Lo mantenemos para comparar con el dataset anterior. Si queda corto o largo, se ve con un barrido.

### 2.6 PSD
Densidad espectral de potencia (Welch, ventanas de 2 s) de `HEO`, `Fp1`, `Pz` y `T7`, en todo el recorte. Las líneas punteadas son los límites de las bandas: δ|θ (4 Hz), θ|α (8), α|β (13) y β|γ (30).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for n in [CANAL_EOG, "Fp1", "Pz", "T7"]:
    f, P = welch(DATOS[n], FS, nperseg=2 * FS)
    ax.semilogy(f, P, label=n)
for f_b, nombre in [(4, "δ|θ"), (8, "θ|α"), (13, "α|β"), (30, "β|γ")]:
    ax.axvline(f_b, color="gray", lw=0.6, ls=":")
ax.set_xlabel("f [Hz]")
ax.set_ylabel("PSD [µV²/Hz]")
ax.set_title("PSD (Welch, ventanas de 2 s) de todo el recorte")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

### Qué mirar: PSD
- **`HEO` tiene casi toda su potencia por debajo de ~10 Hz:** entre 1 y 5 Hz, 100 veces más que el EEG. Es justo donde están δ y θ del EEG. Por eso no sirve un pasa-altos y hace falta la cancelación adaptativa.
- **`Pz` y `T7` tienen un pico en ~10 Hz (α) y otro en ~18 Hz** (β, típico de la corteza motora). `Fp1` tiene menos α. Esas son las partes del EEG que el filtro **no** tiene que tocar.
- **En 50 Hz hay un pozo, no un pico:** la línea ya viene filtrada con un notch. No sabíamos qué filtros traía el `.bdf`; ahora sabemos que al menos este. No hace falta agregar un notch.
- **No hay EMG evidente.** Arriba de ~25 Hz los cuatro canales tienen la misma PSD, incluso `HEO`, y `T7` (un temporal, cerca de los músculos de la mandíbula) no muestra potencia extra.

### 2.7 ¿HEO trae algo más que parpadeo? (fuga de EEG en la referencia)
Este es el riesgo del método (notebook anterior, §1.3): si la referencia trae EEG, el filtro lo puede restar.

La celda mide la correlación entre `HEO` y cada canal en 0.5–15 Hz, **por separado**:
- en las ventanas de parpadeo (±0.5 s alrededor de cada marca);
- en el resto.

Si fuera de los parpadeos la correlación es ~0, `HEO` no comparte EEG con los canales, y el filtro no tiene de dónde restar EEG bueno.

In [ ]:
MARGEN = int(0.5 * FS)                             # Margen alrededor de cada parpadeo [muestras]
con_parpadeo = np.zeros(N_MUESTRAS, bool)
for k in PARPADEOS:
    con_parpadeo[max(k - MARGEN, 0):k + MARGEN] = True
limpio = ~con_parpadeo                             # Muestras a más de 0.5 s de cualquier parpadeo

sos_lp15 = butter(4, 15, "low", fs=FS, output="sos")
eog_lp = sosfiltfilt(sos_lp15, DATOS[CANAL_EOG])
corr_fuera = pd.Series({n: np.corrcoef(eog_lp[limpio], sosfiltfilt(sos_lp15, DATOS[n])[limpio])[0, 1]
                        for n in CANALES_BUENOS + ["M1", "M2"]})
corr_dentro = pd.Series({n: np.corrcoef(eog_lp[con_parpadeo], sosfiltfilt(sos_lp15, DATOS[n])[con_parpadeo])[0, 1]
                         for n in CANALES_BUENOS + ["M1", "M2"]})
fuga = pd.DataFrame({"ρ con parpadeos": corr_dentro, "ρ sin parpadeos": corr_fuera})

print(f"Muestras sin parpadeo (±{MARGEN / FS} s): {limpio.mean() * 100:.0f} %\n")
print("Correlación HEO ↔ canal (0.5-15 Hz):")
print(fuga.loc[["Fp1", "Fp2", "F7", "F8", "Fz", "Cz", "Pz", "O1", "T7", "T8", "M1", "M2"]].round(2).to_string())
print(f"\nMediana en los {len(CANALES_BUENOS)} canales buenos: con parpadeos {corr_dentro[CANALES_BUENOS].median():.2f}, "
      f"sin parpadeos {corr_fuera[CANALES_BUENOS].median():.2f}")

### Qué mirar: fuga de EEG en HEO
- **Con parpadeos, ρ es alta adelante:** 0.92 en `Fp1`/`Fp2`, 0.72 en `F7`. Baja hacia atrás: 0.17 en `Pz`, 0.14 en `O1`.
- **Sin parpadeos, ρ ≈ 0:** la mediana sobre los 51 canales buenos es −0.03. Solo `Fp1` y `Fp2` quedan en 0.15–0.17 (un poco de movimiento de ojos, o algo de EEG frontal que también ve `HEO`).
- **Conclusión:** fuera de los parpadeos `HEO` casi no se parece al EEG. Es una buena referencia: el filtro no debería poder restar mucho EEG.
- **El 78 % del tiempo no hay parpadeo** (con el margen de ±0.5 s). O sea, el filtro pasa la mayor parte del tiempo viendo una referencia que no tiene nada que ver con `y`. Eso importa en §3.

### 2.8 ¿Hay ruido que no sea de los ojos? (parte común)
En 2.2 vimos ondas lentas en todos los canales que no están en `HEO`. Para ver si son la misma señal en todos lados, la celda calcula la correlación entre canales en 0.5–3 Hz.
- Usa solo las muestras fuera de los parpadeos, para que el parpadeo no infle la correlación.
- Incluye canales de adelante, del medio, de los costados y de atrás, más `HEO`, `M1` y `M2`.

In [ ]:
sos_lp3 = butter(4, 3, "low", fs=FS, output="sos")
CANALES_COMUN = ["Fp1", "F8", "Cz", "T7", "T8", "Pz", "O1", CANAL_EOG, "M1", "M2"]
lentos = pd.DataFrame({n: sosfiltfilt(sos_lp3, DATOS[n])[limpio] for n in CANALES_COMUN})   # 0.5-3 Hz, sin parpadeos

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(lentos.corr().values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(CANALES_COMUN)), CANALES_COMUN, rotation=45)
ax.set_yticks(range(len(CANALES_COMUN)), CANALES_COMUN)
for i in range(len(CANALES_COMUN)):
    for j in range(len(CANALES_COMUN)):
        ax.text(j, i, f"{lentos.corr().values[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.8, label="ρ")
ax.set_title("Correlación entre canales en 0.5-3 Hz, fuera de los parpadeos")
plt.tight_layout()
plt.show()

comun = np.mean([DATOS[n] for n in CANALES_BUENOS], axis=0)   # Promedio de los canales buenos
print(f"std de la parte común (promedio de los {len(CANALES_BUENOS)} buenos): {comun.std():.1f} µV; "
      f"std de Pz: {DATOS['Pz'].std():.1f} µV")
print(f"Fracción de la potencia de Pz que explica la parte común: "
      f"{np.corrcoef(comun, DATOS['Pz'])[0, 1]**2 * 100:.0f} %")

### Qué mirar: la parte común
- **Todos los canales de EEG se parecen entre sí casi por completo: ρ = 0.83 a 0.96.** Pasa entre `Fp1` (adelante) y `O1` (atrás), y entre `T7` y `T8` (izquierda y derecha). La actividad del cerebro no hace eso: canales a 20 cm deberían parecerse mucho menos.
- **`HEO`, `M1` y `M2` no participan:** ρ ≈ 0 con todos.
- **El promedio de los 51 canales buenos explica el 91 % de la potencia de `Pz`.** Lo que mide `Pz` es casi todo esta parte común.
- **Hipótesis (no comprobada):**
  - Es algo que comparten todos los canales de EEG y no `HEO`. La explicación más simple es la **referencia**: todos los canales de EEG se miden contra el mismo electrodo, y si ese electrodo se mueve, el ruido aparece en todos. Encaja con que las ondas aparezcan sobre todo al mover el brazo.
  - `HEO` no tiene esta parte, así que probablemente se mide contra otra referencia; por ejemplo, bipolar (un electrodo contra otro, los dos cerca del ojo).
- **Para el filtro:** esta parte común es **ruido en `y` que no está en `x`**. El filtro no tiene con qué armarla, así que no la puede sacar y va a quedar en `e`. Para el filtro funciona como si fuera EEG: hace más grande `J_min`.

### 2.9 Cuánta señal hay sin parpadeos
Cuenta los tramos seguidos sin parpadeo (a más de 0.5 s de cualquier marca) y cuántas ventanas de 1, 2, 4 y 8 s entran en ellos sin superponerse.

Todavía no lo usamos. Es lo que habría disponible para un experimento semi-sintético como el de §4 del notebook anterior: EEG "limpio" más el parpadeo de otro momento.

In [ ]:
# Tramos seguidos sin parpadeo
bordes = np.flatnonzero(np.diff(np.r_[0, limpio.astype(int), 0]))
largos = (bordes[1::2] - bordes[0::2]) / FS        # Duración de cada tramo limpio [s]

filas = {L: {"ventanas sin parpadeo": int(np.sum(np.floor(largos / L))),
             "segundos que cubren": int(np.sum(np.floor(largos / L)) * L)} for L in [1, 2, 4, 8]}
print(pd.DataFrame(filas).T.rename_axis("largo [s]").to_string())
print(f"\nTramos limpios: {len(largos)}, mediana {np.median(largos):.1f} s, máximo {largos.max():.1f} s")
print(f"Tiempo entre parpadeos: mediana {np.median(np.diff(PARPADEOS)) / FS:.1f} s")

### Qué mirar: ventanas sin parpadeo
- **Hay 75 tramos limpios**, de 3.2 s de mediana; el más largo dura 9.2 s. El tiempo típico entre parpadeos es 4.2 s.
- **Ventanas que entran:** 223 de 1 s, 91 de 2 s, 24 de 4 s y 1 de 8 s. En el notebook anterior las realizaciones del Monte Carlo eran de 20 s: acá no hay ni una.
- **Si hacemos semi-sintético,** va a tener que ser con ventanas cortas (2 s = 500 muestras), y el filtro tiene que converger en ese tiempo. Además, por 2.8, ese "EEG limpio" lleva adentro la parte común.

### 2.10 ¿La tarea motora mete ruido?
Desvío de algunos canales en cada fase de la tarea, usando solo las muestras sin parpadeo para que los parpadeos no lo inflen.
- **`std T7 > 30 Hz`** es el desvío arriba de 30 Hz en un temporal. Si la persona apretara la mandíbula o tensara el cuello en alguna fase, subiría.
- **La última columna** son los parpadeos por minuto en cada fase.

In [ ]:
sos_hp30 = butter(4, 30, "high", fs=FS, output="sos")
filas = {}
for fase in ["otra", "fijación", "ejecución", "descanso"]:
    m = (FASE == fase) & limpio                    # Solo muestras de esa fase y sin parpadeo
    filas[fase] = {"segundos": m.sum() / FS}
    for n in ["Fp1", "Pz", "C3", "C4", "T7"]:
        filas[fase][f"std {n}"] = DATOS[n][m].std()
    filas[fase]["std T7 > 30 Hz"] = sosfiltfilt(sos_hp30, DATOS["T7"])[m].std()
    filas[fase]["parpadeos/min"] = np.sum(FASE[PARPADEOS] == fase) / (np.sum(FASE == fase) / FS) * 60
print("std [µV] por fase, sin las ventanas de parpadeo:")
print(pd.DataFrame(filas).T.round(1).to_string())

### Qué mirar: desvío por fase
- **Durante la ejecución el desvío sube ~30–40 %** respecto de la fijación: `Fp1` 24 contra 18 µV, `Pz` 28 contra 20 µV, `C3` 24 contra 19 µV. Encaja con las ondas lentas de 2.2, que aparecen al mover el brazo.
  - Ojo: también hay cambios reales del EEG al mover la mano (en `C3`/`C4` baja el ritmo μ), pero son más chicos y en 8–30 Hz.
- **Arriba de 30 Hz no cambia:** 2–3 µV en todas las fases. No hay EMG durante la tarea.
- **"otra" es la más ruidosa** (`Pz` 34 µV), en los ~19 s de bienvenida.
- **Los parpadeos se concentran en el descanso:** 20 por minuto, contra 9 en la fijación. La persona parpadea cuando la tarea se lo permite.

### Conclusiones de §2

**Lo que medimos:**

| Pregunta | Respuesta |
|---|---|
| Canales | 62 de EEG (51 buenos), `HEO`, y `M1`/`M2` raros (10–20 veces el EEG). Sin EOG vertical ni EMG. |
| Parpadeos | 75 en 336 s, marcados en `Blinks` y bien alineados. ~485 µV en `HEO`, ~200 µV en `Fp1`/`Fp2`. |
| `λ` | 0.4 en `Fp1`/`Fpz`/`Fp2`, 0.13–0.3 en el resto de los frontales, ~0.04 atrás. |
| Retardo ojo→electrodo | 0 ms (≤ 1 muestra). |
| `HEO` como referencia | ρ = 0.9 con `Fp1` en los parpadeos y ≈ 0 fuera de ellos: casi sin fuga de EEG. |
| Frecuencias | El EOG está por debajo de 10 Hz, encima de δ/θ. El 50 Hz ya viene filtrado. Sin EMG visible. |
| Ruido que no es de los ojos | Una parte común a todos los canales de EEG (0.5–3 Hz, ρ ≈ 0.9 entre canales), más fuerte al mover el brazo. No está en `HEO`. |
| Señal sin parpadeos | 78 % del tiempo, pero en tramos cortos (mediana 3.2 s). |

**Decisiones que salen de acá, para §3:**
- `y`: `Fp1` y `Fp2`, que es adonde más parpadeo llega; `Pz` como control (`λ ≈ 0.04`).
- `x`: `HEO`, sin más procesamiento que el pasa-altos común.
- `M = 3`: alcanza de sobra, porque no hay retardo.
- Ni referencia de EMG ni notch: no hacen falta.

**Para discutir:** qué hacer con la parte común (2.8).

## §3. El filtro: NLMS con HEO como referencia

> **Dónde estamos:** ya sabemos qué hay en las señales. Ahora corremos el filtro.
> 1. Las funciones: las mismas del notebook anterior, sin cambios.
> 2. Chequeos de cordura: que el código haga lo que dice la teoría.
> 3. Primera corrida sobre los datos reales: `y` = `Fp1`, `Fp2` y `Pz` (control), `x` = `HEO`.
> 4. Pruebas sobre esa corrida: cómo se ve en el tiempo, qué hacen los coeficientes, si el parpadeo desaparece y si se daña el EEG fuera de los parpadeos.
>
> **No hay ground truth:** no sabemos cuál es el EEG limpio. Todo lo de 3.3 en adelante son pruebas indirectas, como en §6 del notebook anterior.

### 3.1 Funciones
Copiadas tal cual de `proyecto1_g4.ipynb`, §3:
- **`armar_regresor(refs, n, M)`:** arma `x(n)` apilando los últimos `M` valores de cada referencia. Con una sola referencia es `[HEO(n), HEO(n−1), HEO(n−2)]`.
- **`nlms(y, refs, M, mu_t, delta)`:** la fórmula de `7. Variantes de LMS.pdf`, `w(n+1) = w(n) + μ̃ x(n) e*(n) / (‖x(n)‖² + δ)`. Devuelve `e`, `ŷ` y la historia de `w`.
- **`matriz_regresores`, `wiener`:** la solución de Wiener `w_o = R⁻¹p` estimada con promedios temporales. También da `J_min`, los autovalores de `R` y el spread `λmax/λmin`.
- **`aportes`:** qué parte de `ŷ` arma cada referencia.
- **`rrmse`, `rrmse_espectral`:** el error relativo `RMS(a−b)/RMS(b)`, en el tiempo y sobre la PSD.

In [ ]:
def armar_regresor(refs, n, M):
    """x(n) = [x1(n), ..., x1(n-M+1), x2(n), ..., x2(n-M+1), ...]^T (ceros antes de n = 0)."""
    x = np.zeros(len(refs) * M)                      # Vector de entrada apilado, largo K*M
    for k, r in enumerate(refs):                     # Recorre cada referencia x_k
        for i in range(M):                           # Recorre los retardos 0, ..., M-1
            if n - i >= 0:                           # Antes de n = 0 la señal vale 0
                x[k * M + i] = r[n - i]              # x_k(n-i) en su lugar del vector apilado
    return x                                         # Devuelve x(n)


def nlms(y, refs, M, mu_t, delta):
    """
    NLMS con referencias apiladas.
    Devuelve:
        e: error (estimación del EEG limpio)
        y_hat: estimación del ruido en el electrodo
        W: W[n] = w(n), coeficientes usados en la muestra n
    """
    N = len(y)                                       # Cantidad de muestras
    w = np.zeros(len(refs) * M)                      # w(0) = 0
    W = np.zeros((N, len(refs) * M))                 # Historia de los coeficientes
    y_hat = np.zeros(N)                              # ŷ(n)
    e = np.zeros(N)                                  # e(n)
    for n in range(N):                               # Recorre las muestras
        x = armar_regresor(refs, n, M)               # Vector de entrada x(n)
        y_hat[n] = w @ x                             # ŷ(n) = w^H(n) x(n) (señales reales: w^H = w^T)
        e[n] = y[n] - y_hat[n]                       # e(n) = y(n) - ŷ(n)
        W[n] = w                                     # Guarda w(n)
        w = w + mu_t * x * e[n] / (x @ x + delta)    # w(n+1) = w(n) + μ̃ x(n) e*(n) / (||x(n)||² + δ)
    return e, y_hat, W


def matriz_regresores(refs, M):
    """Matriz N x (K*M) cuya fila n es x(n)^T."""
    N = len(refs[0])
    X = np.zeros((N, len(refs) * M))
    for k, r in enumerate(refs):
        for i in range(M):
            X[i:, k * M + i] = r[:N - i]
    return X


def wiener(y, refs, M):
    """Solución de Wiener estimada con promedios temporales."""
    X = matriz_regresores(refs, M)
    R = X.T @ X / len(y)                             # R = E{x x^T}
    p = X.T @ y / len(y)                             # p = E{x y}
    w_o = np.linalg.solve(R, p)                      # w_o = R^-1 p
    lam = np.linalg.eigvalsh(R)
    return {"w_o": w_o, "J_min": np.mean((y - X @ w_o)**2), "autovalores": lam,
            "spread": lam.max() / lam.min(), "P_x": np.trace(R)}


def aportes(refs, W, M):
    """Parte de ŷ que arma cada referencia: ŷ_k(n) = w_k(n)^T x_k(n)."""
    X = matriz_regresores(refs, M)
    return [np.sum(W[:, k * M:(k + 1) * M] * X[:, k * M:(k + 1) * M], axis=1) for k in range(len(refs))]


def rrmse(a, b):
    return np.sqrt(np.mean((a - b)**2) / np.mean(b**2))


def rrmse_espectral(a, b):
    _, Pa = welch(a, FS, nperseg=FS)
    _, Pb = welch(b, FS, nperseg=FS)
    return rrmse(Pa, Pb)

### 3.2 Chequeos de cordura
Antes de creerle al filtro sobre datos reales, tres pruebas donde sabemos la respuesta:
- **(a) Caso de libro:** referencia de ruido blanco e `y = s + h ∗ x`, con `h = [0.8, −0.3, 0.1]` conocido. `w(N)` tiene que llegar a `w_o ≈ h`.
- **(b) Lo mismo, pero con el `HEO` real como referencia** y `Pz` como `s`. La respuesta sigue siendo `h`, pero ahora `x` es pasa-bajos: el spread de `R` es grande y los modos lentos tardan (`6. LMS.pdf`, Convergencia).
- **(c) μ̃ > 2 tiene que divergir** (`7. Variantes de LMS.pdf`, NLMS: estable para 0 < μ̃ < 2).

In [ ]:
M = 3
rng = np.random.default_rng(0)

# (a) Caso de libro: referencia blanca, y = s + h * x  ->  w(N) tiene que llegar a w_o
N = 20000
x1 = rng.normal(size=N)
s = 0.1 * rng.normal(size=N)
y = s + lfilter([0.8, -0.3, 0.1], 1, x1)
_, _, W = nlms(y, [x1], M, 0.1, 1e-6)
print("(a) w(N) =", W[-1].round(3), "  w_o =", wiener(y, [x1], M)["w_o"].round(3), "  (h = [0.8, -0.3, 0.1])")

# (b) Con el HEO real como referencia y h conocido: tiene que llegar a h, pero más lento (spread)
x1 = DATOS[CANAL_EOG]
y = DATOS["Pz"] + lfilter([0.8, -0.3, 0.1], 1, x1)
_, _, W = nlms(y, [x1], M, 0.03, 1e-3 * M * np.mean(x1**2))
wi = wiener(y, [x1], M)
print(f"(b) HEO real: w(N) = {W[-1].round(3)}  w_o = {wi['w_o'].round(3)}  spread de R = {wi['spread']:.0f}")

# (c) μ̃ > 2 tiene que divergir
with np.errstate(all="ignore"):
    e, _, _ = nlms(y[:5000], [x1[:5000]], M, 2.5, 1e-3 * M * np.mean(x1**2))
print(f"(c) μ̃ = 2.5: max|e| en los últimos 1000 = {np.max(np.abs(e[-1000:])):.3g} µV  (enorme o nan: diverge)")

### Qué mirar: chequeos de cordura
- **(a) pasa:** `w(N) = [0.776, −0.299, 0.12]` contra `w_o = [0.8, −0.3, 0.099]`. El código está bien.
- **(b) no llega:**
  - `w(N) = [0.35, −0.78, −0.10]` contra `w_o = [0.80, −0.28, 0.12]`, con un spread de 2685 (en el dataset anterior, el EOG solo daba ~300).
  - La respuesta es la misma que en (a); lo único que cambió es la referencia. Aun así, después de 336 s el filtro no llegó a `w_o`.
- **(c) pasa:** diverge (`nan`).
- **Lo de (b) es lo primero para discutir:** con la referencia real, NLMS no llega a `w_o` ni siquiera con `h` conocido. Se vuelve a ver en 3.5.

### 3.3 Primera corrida sobre los datos reales
`procesar(canal)` corre NLMS con `y` = el canal y `x` = `HEO`, y calcula Wiener sobre la misma señal para comparar.

**Metaparámetros:**
- `M = 3`, por 2.5.
- `μ̃ = 0.03`. **Es el valor que elegimos con el dataset anterior, no está elegido para este.** Lo usamos como punto de partida; el barrido viene después de discutir.
- `δ = 10⁻³ · M · P_x`: solo para no dividir por ~0.

**Canales:** `Fp1` y `Fp2` (`λ ≈ 0.4`), y `Pz` como control (`λ ≈ 0.04`).

**Qué da la tabla:**
- `w(N)` (el último) comparado con `w_o`;
- `J_min/P_y`: qué fracción de la potencia de `y` no puede sacar ni el mejor filtro fijo;
- el spread de `R`;
- la correlación de `y` y de `e` con la referencia.

In [ ]:
M = 3                       # Coeficientes por referencia (He et al. 2004)
MU_T = 0.03                 # Punto de partida: el valor elegido con el dataset anterior, todavía no barrido acá
DELTA_REL = 1e-3            # δ = DELTA_REL · M · P_x: solo evita dividir por ~0
CANALES_Y = ["Fp1", "Fp2", "Pz"]   # Los dos con más parpadeo y un control lejos de los ojos


def procesar(canal, mu_t=MU_T, M=M):
    """NLMS con y = canal y x = HEO, más la solución de Wiener para comparar."""
    y, refs = DATOS[canal], [DATOS[CANAL_EOG]]
    delta = DELTA_REL * M * np.mean(refs[0]**2)
    e, y_hat, W = nlms(y, refs, M, mu_t, delta)
    return {"y": y, "e": e, "y_hat": y_hat, "W": W, "wiener": wiener(y, refs, M)}


reales = {n: procesar(n) for n in CANALES_Y}

filas = {}
for n, r in reales.items():
    w_o = r["wiener"]["w_o"]
    filas[n] = {"w(N)": r["W"][-1].round(3), "w_o": w_o.round(3),
                "‖w(N)−w_o‖/‖w_o‖": np.linalg.norm(r["W"][-1] - w_o) / np.linalg.norm(w_o),
                "J_min / P_y": r["wiener"]["J_min"] / np.mean(r["y"]**2),
                "spread": r["wiener"]["spread"],
                "ρ(y, x)": np.corrcoef(r["y"], DATOS[CANAL_EOG])[0, 1],
                "ρ(e, x)": np.corrcoef(r["e"], DATOS[CANAL_EOG])[0, 1]}
print(pd.DataFrame(filas).T.to_string(float_format=lambda v: f"{v:.3g}"))

### Qué mirar: NLMS contra Wiener
- **Wiener, `Fp1` y `Fp2`:** `w_o ≈ [0.32, 0.04, 0.02]`. Casi todo está en el primer coeficiente, o sea, es una ganancia: la suma (≈ 0.39) se parece al `λ ≈ 0.4` del mapa de 2.4.
- **Wiener, `Pz`:** `w_o ≈ 0`.
- **`J_min/P_y`:**
  - 0.43 en `Fp1`: el mejor filtro fijo saca el 57 % de la potencia de `Fp1`;
  - 0.99 en `Pz`: no hay nada que sacar.

  Los dos encajan con 2.4.
- **NLMS no está cerca de `w_o`:**
  - `Fp1`: `w(N) = [0.64, 0.45, 0.52]`, a una distancia de 2.2 veces `‖w_o‖`;
  - `Pz`: `w(N) = [−0.45, −0.48, −0.20]`, cuando debería ser ~0 (a 26 veces `‖w_o‖`).
- **ρ(e, x) pasa de 0.75 a −0.13 en `Fp1`.** Sacó la mayor parte de lo correlacionado con `HEO`, pero queda negativa: restó de más. Con `w_o`, el principio de ortogonalidad dice que daría 0.
- **Ojo con `w(N)`:** es un solo instante. En 3.5 se ve que `w` cambia todo el tiempo, así que no hay que leerlo como "el valor al que llegó".

### 3.4 En el tiempo
Muestra 15 s (de 40 a 55 s) de `y`, `ŷ` y `e` en los tres canales, con los parpadeos marcados.

In [ ]:
T_VER_3 = (40, 55)                                 # Tramo a mirar [s]
i0, i1 = [int((tv - T_INI) * FS) for tv in T_VER_3]

fig, axs = plt.subplots(len(CANALES_Y), 1, figsize=(12, 8), sharex=True)
for ax, (n, r) in zip(axs, reales.items()):
    ax.plot(t[i0:i1], r["y"][i0:i1], label="y (medido)", lw=0.8)
    ax.plot(t[i0:i1], r["y_hat"][i0:i1], label="ŷ (ruido estimado)", lw=0.8)
    ax.plot(t[i0:i1], r["e"][i0:i1], label="e (EEG estimado)", lw=0.8)
    for k in PARPADEOS[(PARPADEOS >= i0) & (PARPADEOS < i1)]:
        ax.axvline(t[k], color="r", lw=0.8, ls="--")
    ax.set_ylabel(f"{n} [µV]")
    ax.grid(True)
axs[0].legend(loc="upper right")
axs[0].set_title(f"NLMS, x = HEO, M = {M}, μ̃ = {MU_T}. Rojo: parpadeos marcados")
axs[-1].set_xlabel("t [s]")
plt.tight_layout()
plt.show()

### Qué mirar: y, ŷ y e en el tiempo
- **En `Fp1` y `Fp2` los parpadeos se van.** Donde `y` tiene picos de 200–300 µV, `ŷ` los copia y `e` queda al nivel del resto.
- **Pero `ŷ` no es cero entre parpadeos:** tiene oscilaciones rápidas de hasta ±100 µV (por ejemplo, entre 52 y 55 s), y `e` las hereda con el signo opuesto. **Entre parpadeos el filtro agrega algo que no estaba en `y`.**
- **Las ondas lentas comunes de 2.8 quedan en `e`** (por ejemplo, 52–53 s). El filtro no las puede sacar porque no están en `HEO`.
- **En `Pz` el filtro empeora la señal:** en cada parpadeo, `ŷ` tiene un pico de −400 µV y `e` uno de +400 µV, cuando `y` tenía ~20 µV.

### 3.5 Los coeficientes en el tiempo
Muestra `w(n)` de los tres coeficientes para cada canal. En línea punteada del mismo color, el `w_o` de Wiener.

Si NLMS convergiera, cada línea se quedaría cerca de su punteada, con un ruido chico alrededor (el desajuste).

In [ ]:
fig, axs = plt.subplots(1, len(CANALES_Y), figsize=(14, 3.8), sharex=True)
for ax, (n, r) in zip(axs, reales.items()):
    for i in range(M):
        linea, = ax.plot(t, r["W"][:, i], label=f"w{i}", lw=0.8)
        ax.axhline(r["wiener"]["w_o"][i], color=linea.get_color(), ls="--")
    ax.set_title(f"y = {n}")
    ax.set_xlabel("t [s]")
    ax.grid(True)
axs[0].set_ylabel("w(n) (línea: NLMS, punteada: w_o de Wiener)")
axs[0].legend()
plt.tight_layout()
plt.show()

### Qué mirar: los coeficientes
- **Los coeficientes saltan entre −6 y +5**, cuando `w_o` está entre 0 y 0.32. En ningún momento de los 336 s se quedan cerca de `w_o`. No es un transitorio inicial: pasa de principio a fin.
- **En `Pz` pasa lo mismo**, aunque ahí `w_o ≈ 0`.
- **Esto explica 3.4:** cuando llega un parpadeo y `w` está en un valor cualquiera, `ŷ = wᵀx` es ese valor multiplicado por 500 µV.
- **No es un error del código:** pasó el chequeo (a). Es lo que hace NLMS con esta referencia. Hay dos sospechosos, sin comprobar:
  1. el spread de 2685, que deja modos que casi no se corrigen;
  2. que NLMS divide por `‖x(n)‖²`, que **entre parpadeos es chico**. Entonces el paso efectivo `μ̃/‖x‖²` se hace enorme justo cuando `x` no trae parpadeo, y el filtro "aprende" del EEG que hay en `e`.

### 3.6 ¿Desaparece el parpadeo? (promedio sincronizado de y y e)
Esta es la prueba principal: el promedio sincronizado de 2.3, ahora sobre `y` y sobre `e`. Si el filtro saca el parpadeo, la curva de `e` tiene que quedar plana.

La tabla da el pico a pico de cada promedio y cuánto bajó, en dB.

In [ ]:
fig, axs = plt.subplots(1, len(CANALES_Y), figsize=(14, 3.8))
filas = {}
for ax, (n, r) in zip(axs, reales.items()):
    py, pe = promedio_sincronizado(r["y"], PARPADEOS), promedio_sincronizado(r["e"], PARPADEOS)
    ax.plot(tau, py, label="y")
    ax.plot(tau, pe, label="e")
    ax.set_title(n)
    ax.set_xlabel("t respecto del parpadeo [s]")
    ax.grid(True)
    filas[n] = {"pico a pico y [µV]": np.ptp(py), "pico a pico e [µV]": np.ptp(pe),
                "reducción [dB]": 20 * np.log10(np.ptp(pe) / np.ptp(py))}
axs[0].set_ylabel("µV")
axs[0].legend()
plt.tight_layout()
plt.show()
print(pd.DataFrame(filas).T.round(1).to_string())

### Qué mirar: el parpadeo promedio en y y en e
- **En `Fp1` y `Fp2` el pico baja de ~210 a ~40 µV pico a pico (−14 dB).** Sacó la mayor parte del parpadeo.
- **Pero no queda plano: queda un pozo de ~−35 µV en t = 0.** El filtro restó de más en el pico, lo mismo que decía ρ(e, x) < 0 en 3.3. En el notebook anterior también restaba de más el EOG.
- **En `Pz` la curva se da vuelta:** `y` tenía +21 µV y `e` tiene −14 µV. El pico a pico baja solo 3 dB, y es porque cambió de signo, no porque el parpadeo se fue.

### 3.7 ¿Se daña el EEG fuera de los parpadeos?
Compara la potencia de `e` con la de `y`, en dB:
- **0 dB:** no cambió;
- **negativo:** el filtro sacó;
- **positivo:** el filtro agregó.

Se mide en tres lugares:
- 0.5–8 Hz **con** parpadeo, donde queremos que baje;
- 0.5–8 Hz **sin** parpadeo, donde queremos que no cambie;
- α (8–13 Hz) sin parpadeo: el EEG que menos queremos tocar.

La última columna es `RRMSE(e, y)` fuera de los parpadeos. Si el filtro no hace nada ahí, tiene que dar ~0.

In [ ]:
sos_08 = butter(4, [F_HP, 8], "bandpass", fs=FS, output="sos")
sos_alfa = butter(4, [8, 13], "bandpass", fs=FS, output="sos")


def potencia(x, sos, mascara):
    """Potencia de x filtrada con sos, solo en las muestras de la máscara."""
    return np.mean(sosfiltfilt(sos, x)[mascara]**2)


def db(a, b):
    return 10 * np.log10(a / b)


filas = {}
for n, r in reales.items():
    filas[n] = {"0.5-8 Hz con parpadeo [dB]": db(potencia(r["e"], sos_08, con_parpadeo), potencia(r["y"], sos_08, con_parpadeo)),
                "0.5-8 Hz sin parpadeo [dB]": db(potencia(r["e"], sos_08, limpio), potencia(r["y"], sos_08, limpio)),
                "α sin parpadeo [dB]": db(potencia(r["e"], sos_alfa, limpio), potencia(r["y"], sos_alfa, limpio)),
                "RRMSE(e, y) sin parpadeo": rrmse(r["e"][limpio], r["y"][limpio])}
print("Cambio de potencia de e respecto de y (0 dB = no cambió; negativo = el filtro sacó):")
print(pd.DataFrame(filas).T.round(2).to_string())

### Qué mirar: daño al EEG
- **Con parpadeo, 0.5–8 Hz casi no baja (−0.3 dB en `Fp1`),** aunque el promedio bajó 14 dB. Una explicación posible: lo que el filtro saca del parpadeo lo devuelve como oscilaciones (3.4).
- **Sin parpadeo, la potencia sube:** +0.8 a +1.1 dB en 0.5–8 Hz, y **+1.8 a +2.2 dB en α**. El filtro **agrega** potencia donde no tendría que tocar nada.
- **`RRMSE(e, y)` sin parpadeo = 0.86:** fuera de los parpadeos, `e` es muy distinta de `y`. Un filtro que solo sacara parpadeos daría ~0.
- **En `Pz`, +13.9 dB con parpadeo:** confirma lo de 3.4, el filtro le mete parpadeos a un canal que casi no los tenía.

### Para discutir

**Lo que salió:**
- **El parpadeo se va del promedio** en `Fp1`/`Fp2` (−14 dB), pero deja un pozo de −35 µV: resta de más.
- **Fuera de los parpadeos el filtro agrega potencia** (+2 dB en α), y `e` difiere mucho de `y` (RRMSE 0.86).
- **Los coeficientes saltan de −6 a +5 todo el tiempo,** lejos de `w_o`. Ni siquiera con `h` conocido (chequeo (b)) llegan a `w_o`.
- **En `Pz` (el control) el filtro mete parpadeos.**

**Preguntas abiertas:**
1. **¿Por qué saltan los coeficientes?** ¿Es el spread (2685) o el paso efectivo `μ̃/‖x‖²` entre parpadeos? Se puede separar mirando cuánto saltan con `M = 1` (que no tiene spread) y con distintos `μ̃`.
2. **¿Qué `μ̃` y qué `M`?** `0.03` y `3` vienen del dataset anterior. Hace falta un barrido, pero ¿sobre qué métrica, si no hay ground truth?
3. **¿NLMS es la elección correcta acá?** Su argumento era "la potencia de `x` cambia mucho por los parpadeos y NLMS lo compensa". Pero acá la referencia pasa el 78 % del tiempo sin parpadeo.
4. **¿Qué hacemos con la parte común a todos los canales (2.8)?** ¿Se saca antes (por ejemplo, restando el promedio de todos los canales), se deja, o se usa como otra referencia?
5. **¿Hacemos semi-sintético?** Hay ventanas limpias de 1–2 s (2.9): cortas para que el filtro converja, y con la parte común adentro.
6. **¿Agregamos EMG?** En 2.6 y 2.10 no se ve EMG en este registro.

**Cómo seguimos:**
- La pregunta 1 se responde en §4.
- Las preguntas 2 y 3, en §5 (semi-sintético) y §6 (datos reales).
- La 4 queda declarada: la parte común no está en `HEO`, así que este filtro no la puede sacar.
- La 6 queda afuera: en este registro no hay EMG.

## §4. Diagnóstico: por qué saltan los coeficientes

> **Dónde estamos:** en §3, NLMS sacó el parpadeo, pero sus coeficientes saltan entre −6 y +5 todo el tiempo, agrega potencia fuera de los parpadeos y le mete parpadeos a `Pz`. Antes de barrer metaparámetros hay que entender por qué. Si no, barreríamos a ciegas.
>
> **Hipótesis:** NLMS divide por `‖x(n)‖²`, y en esta referencia `‖x(n)‖²` cambia muchísimo entre los parpadeos y el resto.
>
> Pasos:
> 1. Medir cuánto cambia `‖x(n)‖²` y qué paso da cada algoritmo.
> 2. Comparar sobre los datos reales tres variantes: NLMS como en §3, NLMS con un `δ` grande y LMS.

### 4.1 La energía de la referencia y el paso efectivo
NLMS es LMS con un paso que cambia en cada muestra (`7. Variantes de LMS.pdf`, NLMS):

`w(n+1) = w(n) + μ(n) x(n) e(n)`,  con `μ(n) = μ̃ / (‖x(n)‖² + δ)`

La celda dibuja `‖x(n)‖²` (en escala logarítmica) y el paso efectivo `μ(n)` de tres variantes, todas con `μ̃ = 0.03`:
- **NLMS con `δ` chico, el de §3:** `δ = 10⁻³·M·P_x`, casi nada.
- **NLMS con `δ` grande:** `δ = 10·M·P_x`.
  - Mientras `‖x‖² < δ`, el denominador queda ≈ `δ`, y el paso es ≈ `μ̃/δ`: constante, como en LMS.
  - Recién cuando `‖x‖²` supera a `δ` empieza a normalizar. Eso pasa cuando `HEO` supera ~200 µV, o sea en la parte alta de un parpadeo.
- **LMS con `μ = μ̃ / (M·P_x)`:** paso fijo.
  - `M·P_x` es la traza de `R`. Dividir por ella deja `μ̃` con el mismo significado que en NLMS.
  - Como `λmax ≤ tr R`, con `μ̃ < 2` se cumple `μ < 2/λmax` (`6. LMS.pdf`, Convergencia), pero solo *en promedio*: más abajo se ve que eso no alcanza.

In [ ]:
DELTA_CHICO = 1e-3          # El δ de §3: solo evita dividir por ~0
DELTA_GRANDE = 10           # Para comparar: δ = 10·M·P_x (se elige de verdad en §5)

P_X = np.mean(DATOS[CANAL_EOG]**2)                                       # Potencia media de la referencia
norma2 = np.convolve(DATOS[CANAL_EOG]**2, np.ones(M))[:N_MUESTRAS]       # ||x(n)||² = x²(n) + x²(n-1) + x²(n-2)
paso = {f"NLMS, δ = {DELTA_CHICO:g}·M·P_x (§3)": MU_T / (norma2 + DELTA_CHICO * M * P_X),
        f"NLMS, δ = {DELTA_GRANDE:g}·M·P_x": MU_T / (norma2 + DELTA_GRANDE * M * P_X),
        "LMS, μ = μ̃ / (M·P_x)": np.full(N_MUESTRAS, MU_T / (M * P_X))}   # Paso efectivo μ(n) de cada algoritmo

i0, i1 = [int((tv - T_INI) * FS) for tv in T_VER]
fig, axs = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True)
axs[0].semilogy(t[i0:i1], norma2[i0:i1], lw=0.7)
axs[0].set_ylabel("||x(n)||² [µV²]")
for nombre, p in paso.items():
    axs[1].semilogy(t[i0:i1], p[i0:i1], lw=0.8, label=nombre)
axs[1].set_ylabel("paso efectivo μ(n) [1/µV²]")
axs[1].legend(fontsize=8, loc="lower right")
for ax in axs:
    for k in PARPADEOS[(PARPADEOS >= i0) & (PARPADEOS < i1)]:
        ax.axvline(t[k], color="r", lw=0.6, ls="--")
    ax.grid(True, which="both", alpha=0.3)
axs[0].set_title(f"Energía de la referencia y paso efectivo (μ̃ = {MU_T}). Rojo: parpadeos")
axs[1].set_xlabel("t [s]")
plt.tight_layout()
plt.show()

p_con, p_sin = np.mean(DATOS[CANAL_EOG][con_parpadeo]**2), np.mean(DATOS[CANAL_EOG][limpio]**2)
print(f"Potencia de HEO: con parpadeo {p_con:.0f} µV², sin parpadeo {p_sin:.0f} µV² ({p_con / p_sin:.0f} veces menos)")
print("\nPaso efectivo mediano [1/µV²]:")
print(pd.DataFrame({nombre: {"sin parpadeo": np.median(p[limpio]), "en el pico del parpadeo": np.median(p[PARPADEOS])}
                    for nombre, p in paso.items()}).T.to_string(float_format=lambda v: f"{v:.2g}"))

### Qué mirar: la energía de la referencia y el paso
- **`‖x(n)‖²` cubre seis órdenes de magnitud.** Va de ~1 µV² a ~10⁶ µV². Los parpadeos son los picos de ~10⁶, y entre parpadeos queda entre 10 y 10⁴. En promedio, la potencia de `HEO` es **52 veces menor fuera de los parpadeos** (338 contra 17 738 µV²).
- **NLMS con `δ` chico (azul) lo copia al revés.** Entre parpadeos da pasos de 8.7·10⁻⁵ (mediana); en el pico del parpadeo, de 3.8·10⁻⁸: **más de 2000 veces menos**.
- **Eso es justo lo contrario de lo que queremos:**
  - el camino ojo→electrodo solo se puede aprender cuando hay parpadeo;
  - entre parpadeos, `e` es casi todo EEG, y para el filtro eso es ruido en el gradiente;
  - NLMS con `δ` chico da sus pasos más grandes justo cuando la referencia no trae información.
- **Con `δ` grande (naranja),** el paso queda casi constante (2.4·10⁻⁷) y solo baja en los picos.
- **LMS (verde)** tiene un paso constante de 2.4·10⁻⁶, 36 veces menor que el de NLMS con `δ` chico entre parpadeos.
- **Esto contradice el argumento de la presentación.** Elegimos NLMS porque "la potencia de `x` cambia mucho con los parpadeos y NLMS lo compensa". Lo compensa, sí, pero acá compensar está mal: el filtro aprende más fuerte cuando la referencia no trae ruido.

### 4.2 LMS, y tres variantes sobre los datos reales
Tres funciones nuevas:
- **`lms()`** es la fórmula de `6. LMS.pdf` tal cual, `w(n+1) = w(n) + μ x(n) e*(n)`. Es igual a `nlms()` sin la división.
- **`correr()`** corre cualquiera de los dos con un `μ̃` comparable: para NLMS, `δ = δ_rel·M·P_x`; para LMS, `μ = μ̃ / (M·P_x)`.
- **`pruebas_reales()`** junta en un diccionario las pruebas de §3, para no repetir el código:
  - `‖w̄ − w_o‖`: distancia entre Wiener y el promedio de `w(n)` en la segunda mitad de la grabación;
  - `std de w`: cuánto se mueven los coeficientes en esa segunda mitad (promedio de los tres);
  - `parpadeo y`, `parpadeo e`: el pico del promedio sincronizado (dentro de ±0.1 s de la marca), medido desde el nivel de base, en `y` y en `e`;
  - las tres de 3.7: potencia en 0.5–8 Hz con parpadeo, α sin parpadeo y `RRMSE(e, y)` sin parpadeo.

In [ ]:
def lms(y, refs, M, mu):
    """
    LMS con referencias apiladas.
    Devuelve:
        e: error (estimación del EEG limpio)
        y_hat: estimación del ruido en el electrodo
        W: W[n] = w(n), coeficientes usados en la muestra n
    """
    N = len(y)                                       # Cantidad de muestras
    w = np.zeros(len(refs) * M)                      # w(0) = 0
    W = np.zeros((N, len(refs) * M))                 # Historia de los coeficientes
    y_hat = np.zeros(N)                              # ŷ(n)
    e = np.zeros(N)                                  # e(n)
    for n in range(N):                               # Recorre las muestras
        x = armar_regresor(refs, n, M)               # Vector de entrada x(n)
        y_hat[n] = w @ x                             # ŷ(n) = w^H(n) x(n) (señales reales: w^H = w^T)
        e[n] = y[n] - y_hat[n]                       # e(n) = y(n) - ŷ(n)
        W[n] = w                                     # Guarda w(n)
        w = w + mu * x * e[n]                        # w(n+1) = w(n) + μ x(n) e*(n)
    return e, y_hat, W


def correr(algoritmo, y, refs, mu_t, M=3, delta_rel=DELTA_CHICO):
    """
    Corre "NLMS" o "LMS" con un μ̃ comparable entre los dos.
    NLMS: δ = delta_rel · M · P_x.  LMS: μ = μ̃ / (M · P_x), o sea μ̃ dividido por tr R.
    """
    traza = M * sum(np.mean(r**2) for r in refs)     # tr R = M · P_x
    with np.errstate(all="ignore"):                  # LMS puede divergir: que dé nan sin avisar
        if algoritmo == "LMS":
            return lms(y, refs, M, mu_t / traza)
        return nlms(y, refs, M, mu_t, delta_rel * traza)


def pruebas_reales(y, e, W, w_o):
    """Las pruebas de §3 (sin ground truth) en un solo diccionario."""
    py, pe = promedio_sincronizado(y, PARPADEOS), promedio_sincronizado(e, PARPADEOS)
    base = slice(0, VENTANA // 2)                    # De -1 a -0.5 s: nivel de base del promedio
    cerca = slice(VENTANA - FS // 10, VENTANA + FS // 10)   # ±0.1 s alrededor del pico marcado
    pico_y, pico_e = py[cerca] - py[base].mean(), pe[cerca] - pe[base].mean()
    mitad = len(W) // 2                              # Segunda mitad de la grabación
    with np.errstate(all="ignore"):
        return {"‖w̄ − w_o‖": np.linalg.norm(W[mitad:].mean(axis=0) - w_o),
                "std de w": W[mitad:].std(axis=0).mean(),
                "parpadeo y [µV]": pico_y[np.argmax(np.abs(pico_y))],
                "parpadeo e [µV]": pico_e[np.argmax(np.abs(pico_e))],
                "0.5-8 Hz con parp. [dB]": db(potencia(e, sos_08, con_parpadeo), potencia(y, sos_08, con_parpadeo)),
                "α sin parp. [dB]": db(potencia(e, sos_alfa, limpio), potencia(y, sos_alfa, limpio)),
                "RRMSE(e, y) sin parp.": rrmse(e[limpio], y[limpio])}

Ahora corre las tres variantes (NLMS con `δ` chico, NLMS con `δ = 10·M·P_x` y LMS) sobre `Fp1` y `Pz`, con `μ̃ = 0.03` (el de §3) y `μ̃ = 0.1`.

In [ ]:
VARIANTES = {"NLMS δ chico": ("NLMS", DELTA_CHICO), "NLMS δ grande": ("NLMS", DELTA_GRANDE), "LMS": ("LMS", None)}

diag, filas = {}, {}
for canal in ["Fp1", "Pz"]:
    w_o = reales[canal]["wiener"]["w_o"]
    for mu_t in [0.03, 0.1]:
        for nombre, (algoritmo, delta_rel) in VARIANTES.items():
            e, _, W = correr(algoritmo, DATOS[canal], [DATOS[CANAL_EOG]], mu_t, M, delta_rel)
            diag[(canal, nombre, mu_t)] = (e, W)
            filas[(canal, mu_t, nombre)] = pruebas_reales(DATOS[canal], e, W, w_o)

tabla_diag = pd.DataFrame(filas).T
tabla_diag.index.names = ["canal", "μ̃", "algoritmo"]
print(tabla_diag.round(2).to_string())

### Qué mirar: las tres variantes sobre los datos reales
- **Con `μ̃ = 0.03`, cambiar solo `δ` arregla casi todo:**

  | `Fp1`, `μ̃ = 0.03` | NLMS `δ` chico | NLMS `δ` grande | LMS |
  |---|---|---|---|
  | std de `w` | 0.42 | **0.02** | 0.10 |
  | parpadeo en `e` (en `y`: 203 µV) | −36 µV (restó de más) | +26 µV | +13 µV |
  | 0.5–8 Hz con parpadeo | −0.3 dB | **−11.9 dB** | −13.1 dB |
  | α sin parpadeo | +1.8 dB (agrega) | −0.9 dB | −0.8 dB |
  | `RRMSE(e, y)` sin parpadeo | 0.86 | **0.31** | 0.37 |

- **En `Pz` (el control)** NLMS con `δ` grande casi no toca la señal: `RRMSE(e, y)` = 0.07, contra 0.87 con `δ` chico. Ya no le mete parpadeos (+0.95 dB en 0.5–8 Hz, contra +13.9 dB).
- **LMS diverge con `μ̃ = 0.1`** (`NaN`), y NLMS no. Ese `μ̃` cumple `μ < 2/λmax` en promedio, pero en el pico de un parpadeo `‖x‖²` es ~60 veces `tr R`, y ahí el paso queda demasiado grande. **Esta es la debilidad de LMS: su estabilidad depende de la potencia de `x`.** NLMS con cualquier `δ ≥ 0` es estable para `0 < μ̃ < 2`, porque `μ(n)·‖x(n)‖² < μ̃`.
- **Lo que queda por resolver:** el parpadeo no se va del todo (+26 µV de 203) y `‖w̄ − w_o‖` no bajó (0.14 a 0.18). Hay que elegir bien `μ̃` y `δ`, y para eso hace falta un caso donde sepamos la respuesta: §5.

### Los coeficientes de las tres variantes
`w(n)` en `Fp1` con `μ̃ = 0.03`. Los tres paneles tienen la misma escala (−2 a 2), para compararlos directamente. La punteada es `w_o`.

In [ ]:
fig, axs = plt.subplots(1, len(VARIANTES), figsize=(14, 3.8), sharey=True)
w_o = reales["Fp1"]["wiener"]["w_o"]
for ax, nombre in zip(axs, VARIANTES):
    _, W = diag[("Fp1", nombre, 0.03)]
    for i in range(M):
        linea, = ax.plot(t, W[:, i], lw=0.7, label=f"w{i}")
        ax.axhline(w_o[i], color=linea.get_color(), ls="--")
    ax.set_title(f"{nombre} (y = Fp1, μ̃ = 0.03)")
    ax.set_xlabel("t [s]")
    ax.grid(True)
axs[0].set_ylabel("w(n) (punteada: w_o)")
axs[0].set_ylim(-2, 2)
axs[0].legend()
plt.tight_layout()
plt.show()

### Qué mirar: los coeficientes
- **NLMS con `δ` chico:** los coeficientes se salen de la escala todo el tiempo. Es el problema de §3.5.
- **NLMS con `δ` grande:** son casi líneas rectas, sin saltos.
- **LMS:** se mueven más que con `δ` grande, sobre todo al principio (hasta ~0.7 cerca de los 50 s), pero 4 veces menos que NLMS con `δ` chico.
- **Ninguno llega a cada `w_o` por separado:** `w0` queda por debajo de 0.32 y `w1`, `w2` por encima de 0.04 y 0.025. Se ve mejor en 6.3, y se explica ahí.
- **Conclusión de §4:** el problema de §3 no era el código ni (solo) el spread. **Es la normalización de NLMS, que con esta referencia da pasos enormes justo entre parpadeos.** Un `δ` grande lo arregla y mantiene la ventaja de NLMS: no diverge.

## §5. Semi-sintético: el algoritmo contra una respuesta conocida

> **Dónde estamos:** en §4 vimos que un `δ` grande arregla NLMS, pero sobre datos reales no sabemos cuál es el EEG limpio. Entonces no podemos decir qué `μ̃` y qué `δ` limpian *mejor*: solo comparamos pruebas indirectas.

### ⚠️ Qué hacemos en esta sección y qué no
**No estamos diciendo que conocemos el EEG real.** El EEG limpio de una grabación real no se puede conocer, y nada de esta sección lo cambia.

Lo que hacemos es **sintetizar una mezcla con pedazos de EEG real**, armada de forma que **nosotros sabemos la respuesta**. No es para saber cómo es el EEG de esta persona. Es para **evaluar el algoritmo en sí mismo**: si converge, a qué, con qué desajuste, con qué `μ̃`, `δ` y `M`. Es el mismo papel que cumplió el chequeo de cordura (a) de §3.2, pero con señales mucho más parecidas a las reales.

La idea es que el algoritmo **no se entera** de que la mezcla es armada: ve un `y` y un `x` con las mismas propiedades que los reales. Si en la mezcla armada no funciona, en la real tampoco. Si funciona, lo comprobamos en los datos reales en §6.

### Cómo se arma (función `realizacion()`)

```
 canal posterior, tramo A ───────────────────────────────► s(n)   ← "EEG": lo elegimos nosotros
                                                              │
 HEO, tramo B ──► η(n) ──► h ──► × λ ──────────────────────(+)──► y(n)  ← el "electrodo"
                    │
                    └──(+ ruido de medición)──► x(n)            ← la referencia
```

| Pieza | De dónde sale | Por qué así |
|---|---|---|
| **`s(n)`** | 20 s de un canal con poco parpadeo (`λ` < 0.06 en el mapa de 2.4), elegido al azar entre 30. Tramo A. | Es EEG real, con su α, su β y la parte común de 2.8. Trae un poco de parpadeo propio (~5 % del de `HEO`), pero de *otro* momento que `η`, así que para el filtro es parte del EEG. |
| **`η(n)`** | 20 s de `HEO`, de un tramo B **que no se superpone** con A. | Parpadeos reales, con su forma, su ritmo y la potencia de `x` cambiando como en §4. Al venir de otro momento, no tienen relación con `s`. |
| **`h`** | `[1, 0.127, 0.078]`: la forma del `w_o` de Wiener en `Fp1` (§3.3). | Para que el camino ojo→electrodo se parezca al real. |
| **`λ`** | Se calcula para que el SNR sea **−1 dB, el que medimos en `Fp1`**: `SNR = 10·log₁₀(J_min / (P_y − J_min))`. | Para que la cantidad de parpadeo sea la de `Fp1`. |
| **`x(n)`** | `η` más ruido blanco 20 dB por debajo. | Una referencia real nunca es exacta. |

**Monte Carlo:** 30 realizaciones, cada una con otro canal para `s` y otros tramos A y B. Las métricas se miden en la segunda mitad de cada tramo, el régimen permanente.
- **`RRMSE(e, s)`:** cuánto se parece la salida al EEG que pusimos. 0 es perfecto; sin filtrar da 1.30.
  - `RRMSE_e_parp` y `RRMSE_e_sin` lo miden por separado en las ventanas de parpadeo de `η` y en el resto.
- **`CC`:** la correlación de Pearson entre `e` y `s`.
- **Desajuste** `ℳ = (J(∞) − J_min)/J_min`, y **bias**, `‖E{w(N)} − w_o‖`.
- **`t_conv`:** cuándo el error de coeficientes `E{‖w(n) − w_o‖}` llega a menos de un 10 % por encima de su valor final.

La celda define todo eso e imprime qué canales hacen de `s`, `h` y el SNR.

In [ ]:
L_TRAMO = 20 * FS                                  # Largo de cada realización: 20 s
N_REAL = 30                                        # Realizaciones del Monte Carlo
CANALES_S = [n for n in CANALES_BUENOS if abs(LAMBDA[n]) < 0.06]   # Canales con poco parpadeo: hacen de s
w_o_fp1 = reales["Fp1"]["wiener"]["w_o"]
H = w_o_fp1 / w_o_fp1[0]                           # Forma del camino ojo -> electrodo: la de w_o en Fp1 (§3.3)
J_MIN_FP1 = reales["Fp1"]["wiener"]["J_min"]
SNR_FP1 = 10 * np.log10(J_MIN_FP1 / (np.mean(DATOS["Fp1"]**2) - J_MIN_FP1))   # P_s / P_ruido estimado en Fp1
SNR_DB = round(SNR_FP1)                            # SNR de la mezcla: el de Fp1
SNR_REF_DB = 20                                    # Ruido de medición agregado a la referencia


def escalar_por_snr(s, eta, snr_db):
    return np.sqrt(np.mean(s**2) / (np.mean(eta**2) * 10**(snr_db / 10)))


def con_ruido(x, snr_db, rng):
    return x + rng.normal(0, np.sqrt(np.mean(x**2) / 10**(snr_db / 10)), len(x))


def suavizar(x, largo=FS // 2):
    """Promedio móvil de medio segundo (en los bordes promedia solo las muestras que hay)."""
    ventana = np.ones(largo)
    return np.convolve(x, ventana, mode="same") / np.convolve(np.ones(len(x)), ventana, mode="same")


def realizacion(rng, snr_db=SNR_DB, h=H, snr_ref_db=SNR_REF_DB):
    """
    Una mezcla y = s + λ·(h ∗ η), con s y η de tramos distintos de la grabación.
    Devuelve s, y, [x] y en qué muestras η tiene un parpadeo.
    """
    canal_s = rng.choice(CANALES_S)                                       # Canal que hace de EEG
    while True:                                                           # Dos tramos que no se superpongan
        i_s, i_eta = rng.integers(0, N_MUESTRAS - L_TRAMO, 2)
        if abs(i_s - i_eta) >= L_TRAMO:
            break
    s = DATOS[canal_s][i_s:i_s + L_TRAMO]                                 # EEG "limpio": tramo A del canal elegido
    eta = DATOS[CANAL_EOG][i_eta:i_eta + L_TRAMO]                         # Ruido de ojos: tramo B de HEO
    ruido_ojo = lfilter(h, 1, eta)                                        # Camino ojo -> electrodo
    lam = escalar_por_snr(s, ruido_ojo, snr_db)                           # λ para el SNR pedido
    y = s + lam * ruido_ojo                                               # El "electrodo" armado
    x = con_ruido(eta, snr_ref_db, rng)                                   # Referencia: HEO con ruido de medición
    return s, y, [x], con_parpadeo[i_eta:i_eta + L_TRAMO]


def montecarlo(algoritmo, mu_t, delta_rel=DELTA_CHICO, M=3, snr_db=SNR_DB, N_real=N_REAL, semilla=1):
    """Monte Carlo sobre realizacion(). algoritmo: "NLMS" o "LMS"."""
    rng = np.random.default_rng(semilla)
    J = np.zeros(L_TRAMO)                            # Curva de aprendizaje J(n) = E{|e(n)|²}
    D = np.zeros(L_TRAMO)                            # Error de coeficientes D(n) = E{||w(n) - w_o||}
    J_c = np.zeros(L_TRAMO)                          # Error de cancelación E{|e(n) - s(n)|²}
    dif_w, J_min, J_c_wiener, P_s, met = [], [], [], [], []
    regimen = np.zeros(L_TRAMO, bool)
    regimen[L_TRAMO // 2:] = True                    # Segunda mitad: régimen permanente
    for _ in range(N_real):
        s, y, refs, parp = realizacion(rng, snr_db)
        wi = wiener(y, refs, M)
        e, _, W = correr(algoritmo, y, refs, mu_t, M, delta_rel)
        with np.errstate(all="ignore"):
            J += e**2 / N_real
            D += np.linalg.norm(W - wi["w_o"], axis=1) / N_real
            J_c += (e - s)**2 / N_real
        dif_w.append(W[-1] - wi["w_o"])
        J_min.append(wi["J_min"])
        e_o = y - matriz_regresores(refs, M) @ wi["w_o"]                      # Lo que deja el mejor filtro fijo
        J_c_wiener.append(np.mean((e_o - s)**2))
        P_s.append(np.mean(s**2))
        con, sin = regimen & parp, regimen & ~parp
        with np.errstate(all="ignore"):
            met.append([rrmse(y[regimen], s[regimen]), rrmse(e[regimen], s[regimen]),
                        rrmse(e[con], s[con]), rrmse(e[sin], s[sin]), np.corrcoef(e[regimen], s[regimen])[0, 1]])
    J_inf = J[regimen].mean()
    J_min = np.mean(J_min)
    D_inf = D[regimen].mean()
    n_conv = np.argmax(D <= 1.1 * D_inf) if D_inf < D[0] else np.nan   # Primera muestra con D(n) a menos de 10 % de su valor final
    met = np.mean(met, axis=0)
    return {"J": J, "D": D, "J_c": J_c, "J_c_wiener": np.mean(J_c_wiener), "J_min": J_min, "P_s": np.mean(P_s), "J_inf": J_inf,
            "desajuste": (J_inf - J_min) / J_min, "t_conv": n_conv / FS,
            "sesgo": np.linalg.norm(np.mean(dif_w, axis=0)),
            "RRMSE_y": met[0], "RRMSE_e": met[1], "RRMSE_e_parp": met[2], "RRMSE_e_sin": met[3], "CC": met[4]}


print(f"Canales que hacen de s ({len(CANALES_S)}): {CANALES_S}")
print(f"h = {H.round(3)}   SNR de Fp1 = {SNR_FP1:.1f} dB -> SNR_DB = {SNR_DB} dB")

### 5.1 Una realización
Para ver qué recibe el filtro: `s` (el "EEG" que pusimos), `y` (la mezcla, con `s` de fondo en azul) y `x` (la referencia). En rojo, las ventanas de parpadeo de `η`.

In [ ]:
s, y, refs, parp = realizacion(np.random.default_rng(3))
tt = np.arange(L_TRAMO) / FS

fig, axs = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axs[0].plot(tt, s, lw=0.7)
axs[0].set_ylabel("s(n) [µV]\n(EEG, tramo A)")
axs[1].plot(tt, y, lw=0.7, color="tab:orange")
axs[1].plot(tt, s, lw=0.5, color="tab:blue", alpha=0.6)
axs[1].set_ylabel("y(n) [µV]\n(mezcla)")
axs[2].plot(tt, refs[0], lw=0.7, color="tab:green")
axs[2].set_ylabel("x(n) [µV]\n(HEO, tramo B)")
for ax in axs:
    ax.fill_between(tt, 0, 1, where=parp, transform=ax.get_xaxis_transform(), color="r", alpha=0.1, lw=0)
    ax.grid(True)
axs[0].set_title(f"Una realización (SNR = {SNR_DB} dB). Rojo: ventanas de parpadeo de η")
axs[-1].set_xlabel("t [s]")
plt.tight_layout()
plt.show()

### Qué mirar: una realización
- **`s` es EEG real, no algo de libro:** ondas lentas de ~±70 µV (la parte común de 2.8) con α encima.
- **`y` es `s` más los parpadeos de otro momento:** los picos de ~250 µV caen donde `x` tiene sus parpadeos (~550 µV). Entre parpadeos, `y` y `s` casi coinciden.
- **Es el mismo problema que en `Fp1`:** parpadeos de ~200 µV sobre un EEG de decenas de µV, y una referencia que la mayor parte del tiempo no trae nada.

### 5.2 Elegir `μ̃` y `δ`: la grilla
Corre el Monte Carlo para NLMS con cinco valores de `δ` (de 10⁻³ a 100 veces `M·P_x`) y siete de `μ̃` (de 0.001 a 1), y para LMS con los mismos `μ̃`. Cada casilla es un Monte Carlo de 30 realizaciones (unos 2 minutos en total). El color es `RRMSE(e, s)`: más claro es mejor.

In [ ]:
MUS = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]
DELTAS = [1e-3, 0.1, 1, 10, 100]

grilla = {(f"NLMS δ = {d:g}·M·P_x", mu): montecarlo("NLMS", mu, d) for d in DELTAS for mu in MUS}
grilla.update({("LMS", mu): montecarlo("LMS", mu) for mu in MUS})

rr = pd.Series({k: r["RRMSE_e"] for k, r in grilla.items()}).unstack()
rr = rr.loc[[f"NLMS δ = {d:g}·M·P_x" for d in DELTAS] + ["LMS"]].replace(np.inf, np.nan)   # Filas en orden de δ
fig, ax = plt.subplots(figsize=(8, 4.5))
im = ax.imshow(rr.values, cmap="viridis_r", vmin=0, vmax=1.2)
ax.set_xticks(range(len(rr.columns)), [f"{m:g}" for m in rr.columns])
ax.set_yticks(range(len(rr.index)), rr.index)
for i in range(rr.shape[0]):
    for j in range(rr.shape[1]):
        ax.text(j, i, "diverge" if np.isnan(rr.values[i, j]) or rr.values[i, j] > 10 else f"{rr.values[i, j]:.2f}",
                ha="center", va="center", fontsize=8, color="k" if rr.values[i, j] < 0.6 else "w")
ax.set_xlabel("μ̃")
ax.set_title(f"RRMSE(e, s) en régimen ({N_REAL} realizaciones, M = {M}, SNR = {SNR_DB} dB). Sin filtrar: "
             f"{grilla[('LMS', MUS[0])]['RRMSE_y']:.2f}", fontsize=9)
fig.colorbar(im, ax=ax, shrink=0.8, label="RRMSE(e, s)")
plt.tight_layout()
plt.show()

print("Mejor μ̃ de cada fila:")
print(pd.DataFrame({"μ̃": rr.idxmin(axis=1), "RRMSE": rr.min(axis=1)}).round(3).to_string())

### Qué mirar: la grilla
- **NLMS con `δ` chico (la fila de §3) no sirve para ningún `μ̃`.** Lo mejor que logra es 0.92, contra 1.30 sin filtrar. Con `μ̃ = 0.03`, el de §3, da 1.25: casi no limpia. Ahora lo sabemos con una respuesta conocida, no solo con pruebas indirectas.
- **Cuanto más grande `δ`, mejor:** el mejor de cada fila baja de 0.92 a 0.62, 0.51, 0.27 y 0.22.
- **La grilla tiene una diagonal.** Con `δ` grande, lo que importa es el paso entre parpadeos, `μ̃/δ`. Por eso el óptimo se corre a la derecha cuando `δ` crece: `δ = 10` con `μ̃ = 0.03` y `δ = 100` con `μ̃ = 0.3` dan casi el mismo `μ̃/δ` (3·10⁻³).
- **LMS da lo mejor (0.20 con `μ̃ = 0.001`), pero diverge desde `μ̃ = 0.03`.** Es lo de §4: su estabilidad depende de los picos de `‖x‖²`. Además, su margen es chico: entre lo mejor (0.001) y la divergencia (0.03) hay un factor 30.
- **NLMS con `δ = 100·M·P_x` y `μ̃ = 0.3` da 0.22, y no diverge con ningún `μ̃` de la grilla.**
- **Elegimos NLMS con `δ = 100·M·P_x` y `μ̃ = 0.3`.** En la práctica es LMS (`μ = μ̃/δ`) con un seguro: si llega un parpadeo enorme (`HEO` > ~650 µV), normaliza y no diverge. Cuesta 0.02 de RRMSE contra el mejor LMS.
- **¿Por qué no `δ` todavía más grande?** Porque con `δ` → ∞ NLMS *es* LMS con `μ = μ̃/δ`, y el seguro solo actúa en parpadeos cada vez más grandes. `δ = 100` todavía protege y ya está en el óptimo.

### 5.3 Curva de aprendizaje, desajuste y bias
Compara tres configuraciones, cada una en su `μ̃`:
- **NLMS con `δ` chico y `μ̃ = 0.03`:** lo de §3;
- **LMS con `μ̃ = 0.001`:** el mejor de su fila;
- **NLMS con `δ = 100·M·P_x` y `μ̃ = 0.3`:** el elegido.

**Izquierda, la curva de aprendizaje.** No es `J(n) = E{|e(n)|²}` sino **`E{|e(n) − s(n)|²}`, el error de cancelación**: lo que quedó del parpadeo más lo que se le sacó al EEG.
- Con `J(n)` no se ve nada: `J = P_s + (error de cancelación)`, y `P_s` (el EEG, ~70 µV pico) es mucho más grande que el error y sube y baja con las ondas lentas.
- Podemos restarlo porque conocemos `s`, que es justamente para lo que armamos la mezcla.
- La punteada es lo que deja Wiener, el mejor filtro fijo.

**Derecha, el error de coeficientes `E{‖w(n) − w_o‖}`:** cómo se acerca `w` al óptimo.

In [ ]:
MU_LMS = 0.001                                     # Mejor μ̃ de LMS en la grilla
MU_ELEGIDO, DELTA_ELEGIDO = 0.3, 100               # NLMS elegido: el mejor de la grilla
CONFIGS = {"NLMS δ chico (§3)": ("NLMS", MU_T, DELTA_CHICO),
           "LMS": ("LMS", MU_LMS, None),
           "NLMS δ grande (elegido)": ("NLMS", MU_ELEGIDO, DELTA_ELEGIDO)}

res_cfg = {nombre: montecarlo(alg, mu, d) for nombre, (alg, mu, d) in CONFIGS.items()}

tt = np.arange(L_TRAMO) / FS
fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
for nombre, r in res_cfg.items():
    axs[0].plot(tt, 10 * np.log10(suavizar(r["J_c"]) / r["P_s"]), label=nombre)
    axs[1].plot(tt, r["D"], label=nombre)
axs[0].axhline(10 * np.log10(res_cfg["LMS"]["J_c_wiener"] / res_cfg["LMS"]["P_s"]), color="k", ls="--",
               label="Wiener (filtro fijo)")
axs[0].set_ylabel("E{|e(n) − s(n)|²} / P_s [dB] (suavizado 0.5 s)")
axs[0].set_title("Curva de aprendizaje: error de cancelación")
axs[1].set_ylabel("E{‖w(n) − w_o‖}")
axs[1].set_title("Error de coeficientes")
for ax in axs:
    ax.set_xlabel("t [s]")
    ax.grid(True)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

cols = ["RRMSE_y", "RRMSE_e", "RRMSE_e_parp", "RRMSE_e_sin", "CC", "desajuste", "sesgo", "t_conv"]
print(pd.DataFrame({n: {c: r[c] for c in cols} for n, r in res_cfg.items()}).T.round(3).to_string())

### Qué mirar: curvas y tabla
- **NLMS con `δ` chico (azul) no aprende.**
  - Su error de cancelación queda alrededor de 0 dB (tan grande como el EEG) durante los 20 s.
  - Su error de coeficientes **sube**, de 0.23 a 0.4–0.9: `w` se aleja de `w_o`. Por eso `t_conv` da `NaN`.
  - `RRMSE_e = 1.25`, casi igual que sin filtrar (1.30), con desajuste 55 %.
- **LMS y el elegido aprenden casi igual:**
  - el error de cancelación baja de ~0 dB a −15/−20 dB y hacia los 10–12 s llega a lo que deja Wiener (la punteada, −18.5 dB);
  - el error de coeficientes baja de 0.23 a ~0.10 y se queda ahí.
- **El elegido converge un poco más rápido:** `t_conv` 5.9 s contra 8.7 s.
- **El resto de la tabla:**
  - `RRMSE_e` 0.22 contra 0.20;
  - `CC` 0.97 en los dos;
  - sin parpadeo, `RRMSE_e_sin` 0.12 y 0.10: el filtro casi no daña el EEG donde no hay nada que sacar;
  - con NLMS con `δ` chico, `RRMSE_e_sin` = 0.77: daña el EEG justo donde no hay parpadeo. **Esto es "noise amplification" al revés:** NLMS amplifica el ruido del gradiente cuando `‖x‖` es chico.
- **El bias no baja de ~0.09–0.10.** El filtro converge en la *suma* de los coeficientes, que es la ganancia, pero no en cada uno: es el **spread de autovalores** (~280 con `M = 3`). Se ve en 5.4 y en 6.3.
- **El desajuste da negativo (−11 % y −13 %):** el error queda por debajo de `J_min`. Es lo mismo que en el notebook anterior (§4, "¿Cómo puede `J(n)` quedar por debajo de `J_min`?"): un filtro que varía en el tiempo aprovecha correlaciones de casualidad entre `s` y `x` en ventanas cortas. Como no es un desajuste real, lo que manda acá es `RRMSE(e, s)`.

### 5.4 ¿Qué `M`?
Con el elegido, barre `M` = 1, 2, 3, 5 y 8. También da el spread de `R` para cada `M`.

In [ ]:
MS = [1, 2, 3, 5, 8]
alg, mu, d = CONFIGS["NLMS δ grande (elegido)"]
res_M = {m: montecarlo(alg, mu, d, M=m) for m in MS}
spread_M = {m: wiener(*realizacion(np.random.default_rng(3))[1:3], m)["spread"] for m in MS}

tabla_M = pd.DataFrame({m: {"RRMSE_e": r["RRMSE_e"], "RRMSE_e_parp": r["RRMSE_e_parp"], "RRMSE_e_sin": r["RRMSE_e_sin"],
                            "CC": r["CC"], "desajuste": r["desajuste"], "t_conv": r["t_conv"], "spread": spread_M[m]}
                        for m, r in res_M.items()}).T
tabla_M.index.name = "M"
print(tabla_M.round(3).to_string())

### Qué mirar: el barrido de M
- **De 1 a 5 casi no cambia:** `RRMSE_e` 0.24, 0.22, 0.22 y 0.25. Con `M = 8` empeora a 0.31.
- **El spread crece con `M`:** 1, 147, 284, 489 y 792. Cada coeficiente más es un modo lento más (`6. LMS.pdf`, modos). Con `M = 8` el filtro tiene más cosas que aprender, más lento, en tramos de 20 s.
- **Que `M = 1` ande casi igual confirma 2.5:** el camino ojo→electrodo es casi una ganancia pura. Con `M = 1` no hay spread, y aun así el resultado es casi el mismo.
- **Dejamos `M = 3`:** es el mejor junto con `M = 2`, coincide con He et al. (2004) y permite comparar con el notebook anterior.

### 5.5 ¿Y si hay más o menos parpadeo? (barrido de SNR)
Las tres configuraciones de 5.3 con SNR de −10 a +10 dB. −10 dB es un electrodo con mucho parpadeo; +10 dB, uno con poco. La línea negra es sin filtrar.

In [ ]:
SNRS = [-10, -5, 0, 5, 10]
res_snr = {(nombre, snr): montecarlo(alg, mu, d, snr_db=snr) for nombre, (alg, mu, d) in CONFIGS.items() for snr in SNRS}

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(SNRS, [res_snr[(list(CONFIGS)[0], snr)]["RRMSE_y"] for snr in SNRS], "k--o", label="sin filtrar (y)")
for nombre in CONFIGS:
    ax.plot(SNRS, [res_snr[(nombre, snr)]["RRMSE_e"] for snr in SNRS], "-o", label=nombre)
ax.set_xlabel("SNR de la mezcla [dB]")
ax.set_ylabel("RRMSE(·, s)")
ax.set_yscale("log")
ax.set_title(f"Error relativo contra el SNR ({N_REAL} realizaciones, M = {M})")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

print(pd.Series({k: r["RRMSE_e"] for k, r in res_snr.items()}).unstack().round(3).to_string())

### Qué mirar: el barrido de SNR
- **NLMS con `δ` chico queda en ~1.25 para cualquier SNR.** Con SNR ≥ 0 dB es **peor que no filtrar**. Lo que agrega no depende de cuánto parpadeo haya, porque viene de los pasos grandes entre parpadeos.
- **LMS y el elegido mejoran en todos los SNR:**
  - con −10 dB, de 3.6 a 0.36–0.43;
  - con +10 dB, de 0.36 a 0.14–0.19.
- **El elegido es mejor con mucho parpadeo** (−10 dB: 0.36 contra 0.43), y **LMS con poco** (+10 dB: 0.14 contra 0.19). Con mucho parpadeo, el seguro de NLMS actúa más seguido.
- **Ninguno baja de ~0.14–0.19:** es el piso que dejan el ruido de la referencia (20 dB) y el bias de los coeficientes.

### Conclusiones de §5
- **NLMS con `δ` chico no sirve con esta referencia, ni siquiera con la respuesta conocida:** `RRMSE` 1.25 contra 1.30 sin filtrar, y peor que no filtrar con SNR ≥ 0 dB.
- **Lo que hace falta es que el filtro dé pasos chicos entre parpadeos.** LMS lo hace naturalmente, y NLMS con un `δ` grande también.
- **Elegido: NLMS con `δ = 100·M·P_x`, `μ̃ = 0.3`, `M = 3`.**
  - `RRMSE` 0.22 (sin filtrar, 1.30), `CC` 0.97, converge en ~6 s.
  - Es casi tan bueno como el mejor LMS (0.20), pero **no diverge para ningún `μ̃` < 2**, mientras que LMS diverge con un `μ̃` solo 30 veces más grande que su óptimo.
- **Esto cambia el argumento a favor de NLMS:** no es "compensa los cambios de potencia de `x`" (eso acá es malo), sino **"da un LMS con estabilidad garantizada"**.

## §6. De vuelta a los datos reales

> **Dónde estamos:** en §5 elegimos NLMS con `δ = 100·M·P_x`, `μ̃ = 0.3` y `M = 3`, usando mezclas armadas. Ahora la pregunta que importa: **¿sobre la grabación cruda, saca el parpadeo sin dañar el EEG?** De nuevo sin ground truth, con las pruebas de §3.
>
> Comparamos cuatro filtros sobre `Fp1`, `Fp2` y `Pz`:
> - **NLMS con `δ` chico (§3):** el punto de partida.
> - **LMS** con `μ̃ = 0.001`.
> - **NLMS con `δ` grande (elegido).**
> - **Wiener (fijo):** `w_o` calculado con toda la grabación y aplicado igual en todas las muestras. **Es la referencia contra la que comparar:** es lo mejor que puede hacer un filtro que no cambia. Si el adaptativo queda cerca de Wiener, hace bien su trabajo.

### 6.1 La tabla de pruebas
`wiener_fijo()` aplica `w_o` a toda la señal. Después corre los cuatro filtros sobre los tres canales y arma la tabla de `pruebas_reales()` (§4.2), más `ρ(e, x)`, que según el principio de ortogonalidad debería dar 0 (`6. LMS.pdf`, Convergencia, 2º supuesto).

In [ ]:
def wiener_fijo(y, refs, M):
    """El mejor filtro fijo: e = y - w_o^T x(n), con w_o de toda la grabación."""
    w_o = wiener(y, refs, M)["w_o"]
    y_hat = matriz_regresores(refs, M) @ w_o
    return y - y_hat, y_hat, np.tile(w_o, (len(y), 1))


finales, filas = {}, {}
for canal in CANALES_Y:
    y, refs = DATOS[canal], [DATOS[CANAL_EOG]]
    w_o = reales[canal]["wiener"]["w_o"]
    for nombre, (alg, mu, d) in CONFIGS.items():
        finales[(canal, nombre)] = correr(alg, y, refs, mu, M, d)
    finales[(canal, "Wiener (fijo)")] = wiener_fijo(y, refs, M)
    for nombre in list(CONFIGS) + ["Wiener (fijo)"]:
        e, _, W = finales[(canal, nombre)]
        filas[(canal, nombre)] = pruebas_reales(y, e, W, w_o) | {"ρ(e, x)": np.corrcoef(e, refs[0])[0, 1]}

tabla_final = pd.DataFrame(filas).T
tabla_final.index.names = ["canal", "filtro"]
print(tabla_final.round(2).to_string())

### Qué mirar: la tabla de pruebas en datos reales
- **El elegido queda casi igual que Wiener en todo:**

  | `Fp1` | NLMS `δ` chico (§3) | LMS | **Elegido** | Wiener |
  |---|---|---|---|---|
  | std de `w` | 0.42 | 0.01 | 0.02 | 0 |
  | parpadeo en `e` (en `y`: 203 µV) | −36 µV | +25 µV | **+22 µV** | +12 µV |
  | 0.5–8 Hz con parpadeo | −0.3 dB | −13.2 dB | **−13.0 dB** | −14.2 dB |
  | α sin parpadeo | +1.8 dB | −1.0 dB | **−1.0 dB** | −1.0 dB |
  | `RRMSE(e, y)` sin parpadeo | 0.86 | 0.31 | **0.31** | 0.32 |
  | `ρ(e, x)` | −0.13 | 0.02 | **0.01** | 0.00 |

- **El pico del parpadeo baja de 203 a 22 µV (−19 dB) en `Fp1`,** y de 209 a 23 µV en `Fp2`. Wiener deja 12–14 µV.
- **`Pz` (el control) casi no cambia:** `RRMSE(e, y)` = 0.07 sin parpadeo, α −0.03 dB. Su parpadeo chico baja de 23 a 9 µV, igual que con Wiener. Es lo esperable con `λ ≈ 0.04`.
- **Dos cosas que hace también Wiener, así que no son culpa del adaptativo:**
  1. **α baja 1 dB en `Fp1`/`Fp2`.** Lo hace el mejor filtro fijo, así que viene de los datos: `HEO` comparte algo con `Fp1` fuera de los parpadeos (ρ = 0.15–0.17 en 2.7). Puede ser EEG frontal que también ve `HEO` (fuga: estaríamos sacando EEG bueno) o movimientos de ojos chicos (sería ruido bien sacado). **Sin ground truth no lo podemos separar.**
  2. **`RRMSE(e, y)` sin parpadeo = 0.31,** por lo mismo: el filtro también saca lo que comparten `Fp1` y `HEO` fuera de los parpadeos.

### 6.2 En el tiempo
El mismo tramo de §3.4 (40–55 s): `y` en gris, `e` de NLMS con `δ` chico en azul, y `e` del elegido en naranja.

In [ ]:
COLOR = {"y": "0.6", "NLMS δ chico (§3)": "tab:blue", "LMS": "tab:green", "NLMS δ grande (elegido)": "tab:orange",
         "Wiener (fijo)": "k"}
i0, i1 = [int((tv - T_INI) * FS) for tv in T_VER_3]
fig, axs = plt.subplots(len(CANALES_Y), 1, figsize=(12, 8), sharex=True)
for ax, canal in zip(axs, CANALES_Y):
    ax.plot(t[i0:i1], DATOS[canal][i0:i1], lw=0.8, color=COLOR["y"], label="y (medido)")
    ax.plot(t[i0:i1], finales[(canal, "NLMS δ chico (§3)")][0][i0:i1], lw=0.7, color=COLOR["NLMS δ chico (§3)"], label="e, NLMS δ chico (§3)")
    ax.plot(t[i0:i1], finales[(canal, "NLMS δ grande (elegido)")][0][i0:i1], lw=0.9, color=COLOR["NLMS δ grande (elegido)"],
            label="e, NLMS δ grande (elegido)")
    for k in PARPADEOS[(PARPADEOS >= i0) & (PARPADEOS < i1)]:
        ax.axvline(t[k], color="r", lw=0.8, ls="--")
    ax.set_ylabel(f"{canal} [µV]")
    ax.grid(True)
axs[0].legend(loc="upper right", fontsize=8)
axs[0].set_title("El mismo tramo de §3.4, antes y después de cambiar δ")
axs[-1].set_xlabel("t [s]")
plt.tight_layout()
plt.show()

### Qué mirar: en el tiempo
- **Los parpadeos desaparecen** (43.5 s, 47.7 s, 50.7 s): donde `y` tiene picos de 200–300 µV, la naranja sigue de largo.
- **Entre parpadeos, la naranja queda sobre la gris.** Ya no aparecen las oscilaciones rápidas de la azul (por ejemplo, entre 52 y 55 s).
- **En `Pz` la naranja es casi idéntica a `y`,** sin los picos de ±400 µV que metía la azul.
- **Las ondas lentas comunes (2.8) siguen ahí:** el filtro no las puede sacar, como se esperaba.

### 6.3 Los coeficientes del elegido
`w(n)` del elegido en los tres canales. La punteada es `w_o`. Ojo: la escala ahora es mucho más chica que en §3.5.

In [ ]:
fig, axs = plt.subplots(1, len(CANALES_Y), figsize=(14, 3.8), sharex=True)
for ax, canal in zip(axs, CANALES_Y):
    W = finales[(canal, "NLMS δ grande (elegido)")][2]
    w_o = reales[canal]["wiener"]["w_o"]
    for i in range(M):
        linea, = ax.plot(t, W[:, i], lw=0.8, label=f"w{i}")
        ax.axhline(w_o[i], color=linea.get_color(), ls="--")
    ax.set_title(f"y = {canal}")
    ax.set_xlabel("t [s]")
    ax.grid(True)
axs[0].set_ylabel("w(n), NLMS elegido (punteada: w_o)")
axs[0].legend()
plt.tight_layout()
plt.show()

### Qué mirar: los coeficientes del elegido
- **Se mueven poco:** en `Fp1`, entre 0 y 0.3, contra −6 y +5 en §3.5.
- **No llegan a cada `w_o`, pero sí a su suma:** en `Fp1`, `w ≈ [0.2, 0.13, 0.05]` contra `w_o = [0.32, 0.04, 0.025]`. Sumados, 0.38 contra 0.39. **Esto es el spread de autovalores** (2685 en los datos reales; `6. LMS.pdf`, modos):
  - `HEO(n)`, `HEO(n−1)` y `HEO(n−2)` son casi iguales (el parpadeo es lento comparado con 4 ms), así que el autovector del autovalor grande es ≈ `[1, 1, 1]/√3`: la *suma* de los coeficientes. Ese modo es rápido y converge.
  - Las *diferencias* entre coeficientes son los modos de autovalores chicos (58 y 4.6, contra 12 476). Son ~200 a ~2700 veces más lentos, y en 336 s no llegan.
  - **Para la limpieza casi no importa:** con una entrada lenta, lo que cuenta es la suma (la ganancia). Por eso el elegido limpia casi igual que Wiener.
- **En `Pz` los tres coeficientes se mueven juntos, entre −0.1 y +0.1,** alrededor de `w_o ≈ 0.02`. Es el mismo modo rápido, siguiendo un `λ` que es casi 0.

### 6.4 ¿Desaparece el parpadeo? (promedio sincronizado)
La prueba principal de §3.6, con los cuatro filtros. La gris es `y`; la negra punteada, Wiener.

In [ ]:
fig, axs = plt.subplots(1, len(CANALES_Y), figsize=(14, 4))
for ax, canal in zip(axs, CANALES_Y):
    ax.plot(tau, promedio_sincronizado(DATOS[canal], PARPADEOS), color=COLOR["y"], lw=2, label="y")
    for nombre in ["NLMS δ chico (§3)", "LMS", "NLMS δ grande (elegido)", "Wiener (fijo)"]:
        ax.plot(tau, promedio_sincronizado(finales[(canal, nombre)][0], PARPADEOS), lw=1, label=f"e, {nombre}", color=COLOR[nombre],
                ls="--" if nombre == "Wiener (fijo)" else "-")
    ax.set_title(canal)
    ax.set_xlabel("t respecto del parpadeo [s]")
    ax.grid(True)
axs[0].set_ylabel("µV")
axs[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

### Qué mirar: el parpadeo promedio
- **En `Fp1` y `Fp2`, las curvas del elegido, LMS y Wiener quedan casi planas.** Solo queda un pico angosto de ~+20 µV en t = 0, de 203 µV.
  - El pozo de −36 µV de NLMS con `δ` chico (azul) desapareció: ya no resta de más.
  - Wiener deja lo mismo (+12 µV): la forma del parpadeo en `Fp1` no es *exactamente* una copia escalada de la de `HEO`, y con `M = 3` no se puede copiar del todo.
- **En `Pz`:** el pico de +21 µV de `y` baja a ~+8 µV con los tres filtros buenos, y la curva sigue a `y` en todo el resto. La azul se daba vuelta (−14 µV).

### 6.5 ¿Qué frecuencias saca?
PSD de `y` y de `e` en `Fp1` (mucho parpadeo) y `Pz` (casi nada), 0–45 Hz.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, canal in zip(axs, ["Fp1", "Pz"]):
    for nombre in ["y", "NLMS δ chico (§3)", "NLMS δ grande (elegido)", "Wiener (fijo)"]:
        x = DATOS[canal] if nombre == "y" else finales[(canal, nombre)][0]
        f, P = welch(x, FS, nperseg=2 * FS)
        ax.semilogy(f, P, lw=1.5 if nombre == "y" else 0.9, color=COLOR[nombre], label=nombre if nombre == "y" else f"e, {nombre}",
                    ls="--" if nombre == "Wiener (fijo)" else "-")
    ax.set_xlim(0, 45)
    ax.set_title(canal)
    ax.set_xlabel("f [Hz]")
    ax.grid(True, which="both", alpha=0.3)
axs[0].set_ylabel("PSD [µV²/Hz]")
axs[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

### Qué mirar: PSD
- **En `Fp1`, el elegido saca potencia entre ~1 y ~10 Hz:** en 4–7 Hz, la PSD de `e` queda ~10 veces por debajo de la de `y`. Es la banda del parpadeo (2.6). Arriba de ~12 Hz, `e` coincide con `y`: no toca β ni γ.
- **El pico de α (~10 Hz) sigue estando en `e`.** Baja un poco (el −1 dB de 6.1), igual que con Wiener.
- **En `Pz`, `e` y `y` se superponen en todo el rango:** no hay nada que sacar, y no saca nada.
- **NLMS con `δ` chico (azul) agrega potencia en todas las frecuencias,** en `Fp1` y en `Pz`: es el ruido que metían los coeficientes saltando.

## §7. Conclusiones

### En pocas palabras
**Sí: sobre los datos crudos, el filtro adaptativo saca el parpadeo.** El pico baja de ~200 a ~20 µV en `Fp1`/`Fp2` (−19 dB), sin tocar el canal de control ni las frecuencias de arriba de ~12 Hz, y con un resultado casi igual al del mejor filtro fijo (Wiener).

**Pero no funcionó con NLMS tal como lo habíamos planteado.** Con `δ` chico, NLMS da sus pasos más grandes entre parpadeos, cuando la referencia no trae ruido, y mete más ruido del que saca. **Funcionó cuando le pusimos un `δ` grande, que lo hace comportarse como LMS entre parpadeos y le deja la estabilidad garantizada de NLMS.**

### En detalle

**1. El dataset sirve para el problema** (§2).
- Hay parpadeos reales, marcados en `Blinks` y bien alineados (75 en 336 s).
- La referencia `HEO` los ve con ρ = 0.9 y casi no ve EEG fuera de ellos (ρ ≈ 0).
- El camino ojo→electrodo es una ganancia sin retardo: `λ ≈ 0.4` en `Fp1`/`Fp2` y `≈ 0.04` atrás.

**2. NLMS con `δ` chico no anda con esta referencia** (§3, §4).
- **El síntoma:** los coeficientes saltan entre −6 y +5, el parpadeo se resta de más (−36 µV), se agrega potencia fuera de los parpadeos (+2 dB en α), y a `Pz` le mete parpadeos.
- **La causa:** la potencia de `HEO` es 52 veces menor fuera de los parpadeos. NLMS divide por `‖x(n)‖²`, así que entre parpadeos da pasos más de 2000 veces más grandes que en el pico, justo cuando `e` es casi todo EEG y el gradiente es solo ruido.
- **Contra lo que esperábamos:** en la presentación elegimos NLMS porque "compensa los cambios de potencia de `x`". Acá esa compensación es el problema. Es un resultado en sí mismo: la hipótesis de diseño no se cumplió, y sabemos por qué.

**3. Con una mezcla armada sabemos qué funciona y qué no** (§5).
- **NLMS con `δ` chico:** `RRMSE` 1.25 (sin filtrar, 1.30), peor que no filtrar con SNR ≥ 0 dB. Su error de coeficientes *crece*.
- **LMS:** `RRMSE` 0.20, pero diverge con un `μ̃` 30 veces mayor que su óptimo, porque su estabilidad depende de los picos de `‖x‖²`.
- **NLMS con `δ = 100·M·P_x`, `μ̃ = 0.3`:** `RRMSE` 0.22, `CC` 0.97, converge en ~6 s y no diverge para ningún `μ̃ < 2`. **Es el elegido.**
- **`M = 3`** alcanza; de 1 a 5 da casi lo mismo, porque el camino es casi una ganancia.

**4. Sobre los datos crudos, el elegido hace lo que hace Wiener** (§6).

| Prueba (sin ground truth) | Resultado | ¿Pasa? |
|---|---|---|
| Parpadeo promedio en `Fp1`/`Fp2` | 203 → 22 µV (−19 dB); Wiener deja 12 µV | ✔ |
| Potencia 0.5–8 Hz con parpadeo | −13 dB | ✔ |
| Canal de control (`Pz`) | `RRMSE(e, y)` = 0.07; su parpadeo chico baja de 23 a 9 µV, como con Wiener | ✔ |
| Coeficientes estables | se mueven ~0.1, contra ±6 con `δ` chico | ✔ |
| Ortogonalidad `ρ(e, x)` | 0.01 | ✔ |
| Frecuencias de arriba de ~12 Hz | sin cambio | ✔ |
| α fuera de los parpadeos en `Fp1`/`Fp2` | −1 dB, **igual que Wiener** | ? |

**5. Lo que no podemos afirmar, y por qué.**
- **El −1 dB de α en `Fp1`/`Fp2`** también lo hace Wiener, así que viene de los datos y no del algoritmo: `HEO` comparte algo con `Fp1` fuera de los parpadeos. Puede ser EEG frontal (estaríamos sacando EEG bueno) o movimientos de ojos chicos (ruido bien sacado). Sin ground truth no se puede separar.
- **Los coeficientes no llegan a cada `w_o`, solo a su suma** (spread 2685). Para la limpieza casi no importa, porque la entrada es lenta y lo que cuenta es la ganancia. Pero el bias de coeficientes no es cero, y hay que decirlo.
- **Queda un residuo de ~+20 µV en el pico del parpadeo.** Wiener deja ~12 µV, así que la mayor parte es un límite del modelo (`M = 3`, un solo canal de EOG), no del algoritmo.
- **El filtro no saca:**
  - la parte común a todos los canales (2.8), que no está en `HEO`;
  - los movimientos de ojos horizontales, si `HEO` es en realidad vertical (2.2);
  - el EMG: en este registro no hay.
- **Es un solo sujeto y una sola sesión.**

**6. Qué cambia respecto del notebook anterior.**
- En ds008701, el parpadeo ya venía sacado y el ruido de ojos era chico, así que no se podía ver este problema.
- Acá, con parpadeos reales, apareció la limitación de NLMS y la arreglamos con un metaparámetro de la fórmula del curso (`δ`), sin cambiar de algoritmo.
- El argumento para el informe queda así: **NLMS sí, pero por la estabilidad garantizada, no por la normalización. Y `δ` no es "un número chico para no dividir por cero": decide a partir de qué potencia de la referencia el filtro normaliza.**